# **Import Library**

In [ ]:
!pip install ydata-profiling
!pip install catboost

In [ ]:
!pip install catboost

In [ ]:
# general
import pandas as pd
import numpy as np
import math

# visualization
from ydata_profiling import ProfileReport
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

# Feature Selection
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

# Oversampling
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import tensorflow as tf

# Evaluation
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score

from sklearn.model_selection import cross_val_score

# Tuning
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

# **Load Dataset**

In [ ]:
df = pd.read_csv('/content/employee_churn_prediction_updated.csv')
df.head()

In [ ]:
df.info()

- employee_id : ID unik tiap karyawan
- age : umur karyawan dalam tahun
- gender : jenis kelamin karyawan
- education : tingkat pendidikan karyawan (High school, Diploma, Bachelor)
- experience_years : jumlah tahun pengalaman kerja sebelumnya
- monthly target : target kerja bulanan
- target_achievement : persentase pencapaian target
- working_hours_per_week : jam kerja per minggu
- overtime_hours_per_week : jam lembur per minggu
- salary : gaji tahunan (usd)
- commission_rate: persentase komisi
- job_satisfication : skor kepuasan kerja
- work_location : lokasi kerja
- manager_support_score : skor dukungan manager
- company_tenure_years : berapa lama bekerja di perusahaan
- churn : target
- martial_status : status pernikahan
- distance_to_office_km : jarak rumah ke kantor
- churn_period : status periode churn



# **Data Preprocessing & EDA**

## **Describe Data**

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

## **Define Columns**

In [ ]:
# memisahkan kolom numerik dan kategorikal
numerical_columns = df.select_dtypes(include=[np.number]).columns
numerical_columns = numerical_columns.drop(['employee_id', 'churn'])

categorical_columns = df.select_dtypes(include=['object']).columns

print("Numerical Columns:")
print(numerical_columns)
print("\nCategorical Columns:")
print(categorical_columns)

In [ ]:
# The 'categorical_columns' variable was defined before the 'churn_period' column was dropped from df.
# Therefore, it contains 'churn_period' which is no longer in df, leading to a KeyError.
# We need to redefine 'categorical_columns' based on the current state of df.
categorical_columns = df.select_dtypes(include=['object']).columns

# Now, to get unique values for each categorical column, iterate through them.
for col in categorical_columns:
    print(f"Unique values for {col}: {df[col].unique()}")

## **Automatic EDA**

In [ ]:
ProfileReport(df)

## **Check Missing Value & Duplicate Data**

In [ ]:
# Missing Value
missing_values = df.isna().sum()
columns_with_missing_values = missing_values[missing_values > 0]

if not columns_with_missing_values.empty:
    print("Columns with missing values and their counts:")
    print(columns_with_missing_values)
else:
    print("No columns with missing values found.")

In [ ]:
# Duplicate Data
duplicate_rows = df[df.duplicated()]

if not duplicate_rows.empty:
    print("Duplicate rows found:")
    print(duplicate_rows)
    print("Total duplicate rows:", len(duplicate_rows))
else:
    print("No duplicate rows found.")

# **Univariate Analysis**

## **Outlier**

In [ ]:
plt.figure(figsize=(15, 18)) # Adjusted figure size to better accommodate all plots

num_plots = len(numerical_columns)
num_cols = 4  # Number of columns in the subplot grid
num_rows = math.ceil(num_plots / num_cols) # Calculate number of rows needed

for i, column in enumerate(numerical_columns):
    plt.subplot(num_rows, num_cols, i + 1)
    sns.boxplot(x=df[column])
    plt.title(f'Box Plot of {column}')

plt.tight_layout()
plt.show()

## **Numeric Distributions**

In [ ]:
num_plots = len(numerical_columns)
num_cols = 4
num_rows = math.ceil(num_plots / num_cols)

fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 4 * num_rows))

# Flatten the axes array for easy iteration if num_rows > 1
axes = axes.flatten()

for i, col in enumerate(numerical_columns):
  ax = axes[i] # Get the current axis
  for label, grp in df.groupby('churn'):
    # Ensure there's enough data for KDE, otherwise skip or handle differently
    if len(grp[col].dropna().unique()) > 1:
        grp[col].plot.kde(
            ax=ax,
            label=['Churn', 'No Churn'][label], # Changed 'labels' to 'label'
            linewidth=2
        )
    else:
        # For columns with constant values in a group, a simple line might be more appropriate
        # Or, just skip if a KDE is impossible
        if not grp[col].empty:
            ax.axvline(x=grp[col].iloc[0], color=['blue', 'orange'][label], linestyle='--', label=f"{['Churn', 'No Churn'][label]} (Constant)")

  ax.set_title(f'Distribution of {col}')
  ax.set_xlabel(col)
  ax.legend()

# Hide any unused subplots if the grid has more slots than plots
for j in range(len(numerical_columns), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(
    'Numeric Feature Distribution by Churn',
    fontsize=14,
    fontweight='bold'
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust rect for suptitle to prevent overlap
plt.show()

## **Numeric Distributions (Histograms)**

In [ ]:
num_plots = len(numerical_columns)
num_cols = 4
num_rows = math.ceil(num_plots / num_cols)

fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 4 * num_rows))

# Flatten the axes array for easy iteration if num_rows > 1
axes = axes.flatten()

for i, col in enumerate(numerical_columns):
  ax = axes[i] # Get the current axis
  sns.histplot(
      data=df,
      x=col,
      hue='churn',
      kde=False, # Disable KDE for histogram
      palette={0: 'blue', 1: 'orange'},
      ax=ax
  )
  ax.set_title(f'Distribution of {col}')
  ax.set_xlabel(col)
  ax.legend(title='Churn', labels=['Churn', 'No Churn'])

# Hide any unused subplots if the grid has more slots than plots
for j in range(len(numerical_columns), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(
    'Numeric Feature Distribution by Churn (Histograms)',
    fontsize=14,
    fontweight='bold'
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust rect for suptitle to prevent overlap
plt.show()

In [ ]:
def plot_features_by_churn(df, features, plot_type_mapping, num_cols=3, title_prefix="Distribution of"):
    num_rows = math.ceil(len(features) / num_cols)

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(5 * num_cols, 4 * num_rows))
    axes = axes.flatten()

    total_employees = len(df)

    for i, col in enumerate(features):
        ax = axes[i] # Assign the current subplot axis
        # The original code only contained a countplot, so I'll keep that for now.
        # The plot_type_mapping parameter is not utilized in this corrected version
        sns.countplot(data=df, x=col, hue='churn', ax=ax, palette='viridis')
        ax.tick_params(axis='x', labelrotation=45)

        for container in ax.containers:
            for p in container.patches:
                height = p.get_height()
                if height > 0:
                    percentage = (height / total_employees) * 100
                    ax.annotate(f'{int(height)}\n({percentage:.1f}%)',
                                (p.get_x() + p.get_width() / 2., height),
                                ha='center', va='top', fontsize=7, color='white', xytext=(0, -5),
                                textcoords='offset points')

        ax.set_title(f'{col} by Churn Status')
        ax.legend(title='Churn', labels=['Stayed (0)', 'Churned (1)'], loc='upper right')
        ax.set_xlabel(col)
        ax.set_ylabel('Count / Density')

    # Hide any unused subplots if the grid has more slots than plots
    for j in range(len(features), len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

In [ ]:
# Identify categorical features
categorical_features = df.select_dtypes(include='object').columns.tolist()

# Define plot types for categorical features (all countplot for this section)
categorical_plot_type_mapping = {feature: 'countplot' for feature in categorical_features}

# Plot categorical features
plot_features_by_churn(df, categorical_features, categorical_plot_type_mapping, num_cols=2, title_prefix="Categorical")

## **Churn**

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
churn_counts = df['churn'].value_counts()

ax.pie(
    churn_counts,
    labels=['Churn', 'No Churn'],
    autopct='%1.1f%%',
    startangle=90,
    explode=[0, 0.05]
)
ax.set_title(
    'Churn Distribution',
    fontsize=12,
    fontweight='bold'
)
plt.tight_layout()
plt.show()

## **Churn Periode**

In [ ]:
pd.crosstab(df['churn_period'], df['churn'])

In [ ]:
df.groupby(['churn_period'])['company_tenure_years'].agg(['min', 'max', 'mean', pd.Series.mode])

In [ ]:
onboarding_df = df[df['churn_period'] == 'Onboarding']
onboarding_df['company_tenure_years'].value_counts().sort_index()

# **Statistical Analysis**

In [ ]:
from scipy import stats

# --- Numerical Features T-tests ---
print("\n--- T-tests for Numerical Features ---")
for col in numerical_columns:
    # Separate data for churned and non-churned employees
    data_stayed = df[df['churn'] == 0][col]
    data_churned = df[df['churn'] == 1][col]

    # Perform independent samples t-test
    # Handle cases where one group might have no variance or insufficient data
    if len(data_stayed) > 1 and data_stayed.std() > 0 and len(data_churned) > 1 and data_churned.std() > 0:
        t_stat, p_value = stats.ttest_ind(data_stayed, data_churned, equal_var=False) # Assuming unequal variances
        print(f"\nFeature: {col}")
        print(f"  T-statistic: {t_stat:.3f}")
        print(f"  P-value: {p_value:.3f}")

        # Interpret the results
        alpha = 0.05
        if p_value < alpha:
            print(f"  Conclusion: Statistically significant difference in {col} between churned and non-churned employees.")
        else:
            print(f"  Conclusion: No statistically significant difference in {col} between churned and non-churned employees.")

        print(f"  Average {col} for Stayed: {data_stayed.mean():.3f}")
        print(f"  Average {col} for Churned: {data_churned.mean():.3f}")
    else:
        print(f"\nFeature: {col}")
        print("  Skipping T-test due to insufficient variance or data in one of the groups.")

# --- Categorical Features Chi-square Tests ---
print("\n--- Chi-square Tests for Categorical Features ---")
for col in categorical_columns:
    print(f"\nFeature: {col}")
    contingency_table = pd.crosstab(df[col], df['churn'])

    # Perform Chi-square test if there are enough observations
    if contingency_table.min().min() > 0 and contingency_table.shape[0] > 1 and contingency_table.shape[1] > 1:
        chi2, p_value, _, _ = stats.chi2_contingency(contingency_table)
        print(f"  Chi-square statistic: {chi2:.3f}")
        print(f"  P-value: {p_value:.3f}")

        alpha = 0.05
        if p_value < alpha:
            print(f"  Conclusion: Statistically significant association between {col} and churn.")
        else:
            print(f"  Conclusion: No statistically significant association between {col} and churn.")
    else:
        print("  Skipping Chi-square test due to insufficient observations in contingency table or too few categories.")

    print("  Contingency Table:")
    print(contingency_table)

In [ ]:
import pandas as pd

# Initialize a list to store the summary results
summary_data = []

# --- Numerical Features T-tests ---
for col in numerical_columns:
    data_stayed = df[df['churn'] == 0][col]
    data_churned = df[df['churn'] == 1][col]

    if len(data_stayed) > 1 and data_stayed.std() > 0 and len(data_churned) > 1 and data_churned.std() > 0:
        t_stat, p_value = stats.ttest_ind(data_stayed, data_churned, equal_var=False)
        conclusion = "Statistically significant difference" if p_value < 0.05 else "No statistically significant difference"
        summary_data.append({
            'Feature': col,
            'Test': 'Independent Samples T-test',
            'P-value': p_value,
            'Conclusion': conclusion
        })
    else:
        summary_data.append({
            'Feature': col,
            'Test': 'Independent Samples T-test',
            'P-value': 'N/A',
            'Conclusion': 'Skipped (insufficient variance or data)'
        })

# --- Categorical Features Chi-square Tests ---
for col in categorical_columns:
    contingency_table = pd.crosstab(df[col], df['churn'])
    if contingency_table.min().min() > 0 and contingency_table.shape[0] > 1 and contingency_table.shape[1] > 1:
        chi2, p_value, _, _ = stats.chi2_contingency(contingency_table)
        conclusion = "Statistically significant association" if p_value < 0.05 else "No statistically significant association"
        summary_data.append({
            'Feature': col,
            'Test': 'Chi-square Test of Independence',
            'P-value': p_value,
            'Conclusion': conclusion
        })
    else:
        summary_data.append({
            'Feature': col,
            'Test': 'Chi-square Test of Independence',
            'P-value': 'N/A',
            'Conclusion': 'Skipped (insufficient observations or categories)'
        })

# Create DataFrame
summary_df = pd.DataFrame(summary_data)
display(summary_df.round(3))

In [ ]:
churned_df = df[df['churn'] == 1]
stayed_df  = df[df['churn'] == 0]

In [ ]:
churned_df.describe()

Let's compare the mean and std of the employees who stayed and left
- 'target_achievement': Employees who stayed have higher target achievement
- 'working_hours_per_week': Employees who stayed work 3 hours less than the employees who left
- 'job_satisfaction': Employees who stayed is more satisfied with the job
- 'manager_support_score': Employees who stayed got higher support from the manager
- 'company_tenure_years': Employees who stayed tend to have a slightly longer tenure with the company
- 'distance_to_office_km': Employees who stayed live closer to the office, on average

In [ ]:
stayed_df.describe()

# **Multivariate Analysis**

## **Churn Rate**

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(categorical_columns):
  cr = df.groupby(col)['churn'].mean().sort_values(ascending=False)
  cr.plot(
      kind='bar',
      ax=axes[i],
      color=sns.color_palette('viridis', len(cr)),
      edgecolor='black',
      width=0.8
  )
  axes[i].set_title(f'Churn Rate by {col}')
  axes[i].set_xlabel('')
  axes[i].set_ylabel('Churn Rate')
  axes[i].tick_params(axis='x', rotation=0)
  for bar in axes[i].patches:
    axes[i].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{bar.get_height():.1%}',
        ha='center',
        va='bottom',
        fontsize=8
    )
for j in range(i+1, len(axes)):
  axes[j].set_visible(False)
plt.suptitle(
    'Churn Rate by Categorical Features',
    fontsize=14,
    fontweight='bold',
    y=1.01
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
for col in categorical_columns:
    print(f"\n--- Churn Counts by {col} ---")
    churn_by_category = df.groupby([col, 'churn']).size().unstack(fill_value=0)
    # Calculate percentage for better understanding
    churn_by_category['Total'] = churn_by_category.sum(axis=1)
    churn_by_category['Churn_Percentage'] = (churn_by_category[1] / churn_by_category['Total']) * 100
    display(churn_by_category.round(2))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10)) # Increased height for better visibility

axes = axes.flatten()

# Plot for work_location
sns.countplot(data=df, x='work_location', hue='churn', palette='viridis', ax=axes[0])
axes[0].set_title('Work Location Distribution by Churn Status')
axes[0].legend(title='Churn', labels=['Stayed (0)', 'Churned (1)'])

# Plot for marital_status
sns.countplot(data=df, x='marital_status', hue='churn', palette='magma', ax=axes[1])
axes[1].set_title('Marital Status Distribution by Churn Status')
axes[1].legend(title='Churn', labels=['Stayed (0)', 'Churned (1)'])

# Plot for gender
sns.countplot(data=df, x='gender', hue='churn', palette='Pastel1', ax=axes[2])
axes[2].set_title('Gender Distribution by Churn Status')
axes[2].legend(title='Churn', labels=['Stayed (0)', 'Churned (1)'])

# Plot for education
sns.countplot(data=df, x='education', hue='churn', palette='Pastel2', ax=axes[3])
axes[3].set_title('Education Distribution by Churn Status')
axes[3].tick_params(axis='x', rotation=45)
axes[3].legend(title='Churn', labels=['Stayed (0)', 'Churned (1)'])

# Plot for churn_period
sns.countplot(data=df, x='churn_period', hue='churn', palette='coolwarm', ax=axes[4])
axes[4].set_title('Churn Period Distribution by Churn Status')
axes[4].tick_params(axis='x', rotation=45)
axes[4].legend(title='Churn', labels=['Stayed (0)', 'Churned (1)'])

# Hide any unused subplots (only axes[5] will be hidden now)
for j in range(5, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    'Categorical Feature Distributions by Churn Status',
    fontsize=16,
    fontweight='bold',
    y=1.02 # Adjust suptitle position
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust rect for suptitle to prevent overlap
plt.show()

## **Churned Employees Analysis**

In [ ]:
df_churned = df[df['churn'] == 1]

### **Demographics of Churned Employees**

### **Job-Related Factors of Churned Employees**

### **Performance and Financial Data of Churned Employees**

### **Job Satisfaction and Manager Support for Churned Employees**

## **Correlation**

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_corr = df.copy()

for c in df_corr.select_dtypes('object').columns:
  df_corr[c] = LabelEncoder().fit_transform(df_corr[c])
corr = df_corr.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 8}
)
ax.set_title(
    'Feature Correlation Heatmap',
    fontsize=14,
    fontweight='bold'
)
plt.tight_layout()
plt.show()

# **Drop Unnecessary Columns**

In [ ]:
# drop feature
df = df.drop(['employee_id', 'churn_period'], axis=1)

In [ ]:
df.info()

# **Feature Engineering**

## **Overtime Ratio**

In [ ]:
# It could highlight employees with significant overtime relative to their standard hours

df['overtime_ratio'] = (
    df['overtime_hours_per_week'] /
    df['working_hours_per_week']
)

##**Overall Satisfaction**

In [ ]:
# Interaction terms: job_satisfaction & manager_support_score
# It might reveal if high job satisfaction only leads to retention when manager support is also high.
# It might boost performance or help the model find that relationship more easily

df['overall_satisfaction'] = df['job_satisfaction'] * df['manager_support_score']

## **Unachieved Target**

In [ ]:
# It shows the target that haven't been achieved within a month

df['unachieved_target'] = (
    df['monthly_target'] *
    (1 - df['target_achievement'])
)

## **Salary Level**

In [ ]:
#df['salary_level'] = pd.qcut(
 #   df['salary'],
  #  q=3,
   # labels=['Low','Medium','High']
#)

## **Distance Group**

In [ ]:
# It might capture thresholds beyond which distance becomes a significant churn factor

df['distance_group'] = pd.cut(
    df['distance_to_office_km'],
    bins=[0,10,25,50],
    labels=['Near','Medium','Far']
)

# **Define Columns**

In [ ]:
numerical_columns = df.select_dtypes(include=[np.number]).columns
numerical_columns = numerical_columns.drop(['churn'])

categorical_columns = df.select_dtypes(exclude=[np.number]).columns

print("Numerical Columns:")
print(numerical_columns)
print("\nCategorical Columns:")
print(categorical_columns)

# **Train-Test Split**

In [ ]:
X = df.drop('churn', axis=1)
y = df['churn']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print (f'Training set shape: {X_train.shape}')
print (f'Test set shape: {X_test.shape}')

# **Pipeline**

## **Preprocessing**

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, numerical_columns),
    ('cat', cat_pipeline, categorical_columns)
])

## **Feature Selection**

In [ ]:
# Find best k feature
X_encoded = preprocessor.fit_transform(X_train)

k_values = range(5, 31, 5)

scores = []

for k in k_values:
    selector = SelectKBest(
        score_func=f_classif,
        k=k
    )

    X_selected = selector.fit_transform(
        X_encoded,
        y_train
    )

    model = LogisticRegression(
        max_iter=1000
    )

    score = cross_val_score(
        model,
        X_selected,
        y_train,
        cv=5,
        scoring='recall'
    ).mean()

    scores.append(score)

In [ ]:
plt.plot(
    k_values,
    scores,
    marker='o'
)

plt.xlabel("k features")
plt.ylabel("F1 Score")

plt.title("Finding Best k")

plt.show()

In [ ]:
best_k = k_values[scores.index(max(scores))]

print("Best k:", best_k)

In [ ]:
selector = SelectKBest(
    score_func=f_classif,
    k='all'
)

selector.fit(X_encoded, y_train)

# Get feature names after one-hot encoding
# The preprocessor creates a sparse matrix, so we need to get the feature names correctly
feature_names = preprocessor.get_feature_names_out()

feature_scores = pd.DataFrame({
    'Feature': feature_names,
    'Score': selector.scores_
})

feature_scores = feature_scores.sort_values(
    by='Score',
    ascending=False
)

feature_scores

In [ ]:
# # get features names
# get_features = feature_scores.head(best_k)
# selected_features = get_features['Feature'].tolist()
# selected_features

In [ ]:
# from sklearn.preprocessing import FunctionTransformer

# def filter_features(X):
#   if hasattr(X, 'loc'):
#     return X[selected_features]
#   return X

# feature_selection = FunctionTransformer(filter_features)

In [ ]:
feature_selector = SelectKBest(
    score_func=f_classif,
    k=best_k
)

## **Oversampling (SMOTE)**

In [ ]:
#smote = SMOTE(random_state=42)

# **Modeling Pipeline**

## **Pipeline Template**

In [ ]:
def create_pipeline(model):
  pipeline = ImbPipeline([
      ('preprocessor', preprocessor),
      ('feature_selector', feature_selector),
      #('smote', smote),
      ('model', model)
  ])

  return pipeline

## Model 1 - Logistic Regression

In [ ]:
lr_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('feature_selector', feature_selector),
    #('smote', smote),
    ('model', LogisticRegression(
        random_state=42
    ))
])

lr_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_lr = lr_pipeline.predict(X_test)
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr))

#### **Logistic Regression Tuned**

In [ ]:
y_proba_lr_tuned = lr_pipeline.predict_proba(X_test)[:, 1]
y_pred_lr_tuned = (y_proba_lr_tuned >= 0.40).astype(int)

print("Logistic Regression (t=0.40) — Final Model")
print(classification_report(y_test, y_pred_lr_tuned))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr_tuned, cmap='Blues')
plt.title('Logistic Regression (t=0.40)')
plt.show()

#### **GridSearchCV for Logistic Regression**

In [ ]:
# Define parameter grid for Logistic Regression
param_grid_lr = {
    'model__penalty': ['l1', 'l2'],
    'model__C': [0.1, 1, 10]
}

# Create GridSearchCV object
grid_search_lr = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=param_grid_lr,
    scoring='recall',
    cv=5,
    verbose=1,
    n_jobs=-1
)

# Fit GridSearchCV
grid_search_lr.fit(X_train, y_train)

print("Best parameters for Logistic Regression (GridSearchCV):")
print(grid_search_lr.best_params_)
print("Best recall score for Logistic Regression (GridSearchCV):")
print(grid_search_lr.best_score_)

y_pred_lr_grid = grid_search_lr.predict(X_test)
print("\nLogistic Regression (GridSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_lr_grid))

#### **RandomizedSearchCV for Logistic Regression**

In [ ]:
from scipy.stats import uniform

# Define parameter distributions for Logistic Regression
param_dist_lr = {
    'model__penalty': ['l1', 'l2'],
    'model__C': uniform(loc=0.01, scale=100)
}

# Create RandomizedSearchCV object
random_search_lr = RandomizedSearchCV(
    estimator=lr_pipeline,
    param_distributions=param_dist_lr,
    scoring='recall',
    n_iter=20, # Number of parameter settings that are sampled
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit RandomizedSearchCV
random_search_lr.fit(X_train, y_train)

print("Best parameters for Logistic Regression (RandomizedSearchCV):")
print(random_search_lr.best_params_)
print("Best recall score for Logistic Regression (RandomizedSearchCV):")
print(random_search_lr.best_score_)

y_pred_lr_rand = random_search_lr.predict(X_test)
print("\nLogistic Regression (RandomizedSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_lr_rand))

## **Model 2 - Decision Tree**

In [ ]:
dt_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('feature_selector', feature_selector),
    #('smote', smote),
    ('model', DecisionTreeClassifier(
        random_state=42
    ))
])

dt_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_dt = dt_pipeline.predict(X_test)
print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))

#### **GridSearchCV for Decision Tree**

In [ ]:
# Define parameter grid for Decision Tree
param_grid_dt = {
    'model__max_depth': [3, 5, 7, 9],
    'model__min_samples_leaf': [1, 5, 10],
    'model__criterion': ['gini', 'entropy']
}

# Create GridSearchCV object
grid_search_dt = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=param_grid_dt,
    scoring='recall',
    cv=5,
    verbose=1,
    n_jobs=-1
)

# Fit GridSearchCV
grid_search_dt.fit(X_train, y_train)

print("Best parameters for Decision Tree (GridSearchCV):")
print(grid_search_dt.best_params_)
print("Best recall score for Decision Tree (GridSearchCV):")
print(grid_search_dt.best_score_)

y_pred_dt_grid = grid_search_dt.predict(X_test)
print("\nDecision Tree (GridSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_dt_grid))

#### **RandomizedSearchCV for Decision Tree**

In [ ]:
from scipy.stats import randint

# Define parameter distributions for Decision Tree
param_dist_dt = {
    'model__max_depth': randint(3, 15),
    'model__min_samples_leaf': randint(1, 20),
    'model__criterion': ['gini', 'entropy']
}

# Create RandomizedSearchCV object
random_search_dt = RandomizedSearchCV(
    estimator=dt_pipeline,
    param_distributions=param_dist_dt,
    scoring='recall',
    n_iter=20, # Number of parameter settings that are sampled
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit RandomizedSearchCV
random_search_dt.fit(X_train, y_train)

print("Best parameters for Decision Tree (RandomizedSearchCV):")
print(random_search_dt.best_params_)
print("Best recall score for Decision Tree (RandomizedSearchCV):")
print(random_search_dt.best_score_)

y_pred_dt_rand = random_search_dt.predict(X_test)
print("\nDecision Tree (RandomizedSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_dt_rand))

## **Model 3 - Random Forest**

In [ ]:
rf_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('feature_selector', feature_selector),
    #('smote', smote),
    ('model', RandomForestClassifier(
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_pipeline.predict(X_test)
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

#### **GridSearchCV for Random Forest**

In [ ]:
# Define parameter grid for Random Forest
param_grid_rf = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [5, 10, 15],
    'model__min_samples_leaf': [1, 5, 10]
}

# Create GridSearchCV object
grid_search_rf = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid_rf,
    scoring='recall',
    cv=5,
    verbose=1,
    n_jobs=-1
)

# Fit GridSearchCV
grid_search_rf.fit(X_train, y_train)

print("Best parameters for Random Forest (GridSearchCV):")
print(grid_search_rf.best_params_)
print("Best recall score for Random Forest (GridSearchCV):")
print(grid_search_rf.best_score_)

y_pred_rf_grid = grid_search_rf.predict(X_test)
print("\nRandom Forest (GridSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_rf_grid))

#### **RandomizedSearchCV for Random Forest**

In [ ]:
from scipy.stats import uniform

# Define parameter distributions for Random Forest
param_dist_rf = {
    'model__n_estimators': randint(50, 500),
    'model__max_depth': randint(3, 20),
    'model__min_samples_leaf': randint(1, 20),
    'model__min_samples_split': randint(2, 20), # Added min_samples_split for regularization
    'model__max_features': uniform(0.1, 0.9) # max_features acts as regularization
}

# Create RandomizedSearchCV object
random_search_rf = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_dist_rf,
    scoring='recall',
    n_iter=20, # Number of parameter settings that are sampled
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit RandomizedSearchCV
random_search_rf.fit(X_train, y_train)

print("Best parameters for Random Forest (RandomizedSearchCV):")
print(random_search_rf.best_params_)
print("Best recall score for Random Forest (RandomizedSearchCV):")
print(random_search_rf.best_score_)

y_pred_rf_rand = random_search_rf.predict(X_test)
print("\nRandom Forest (RandomizedSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_rf_rand))

## **Model 4 - XGBoost**

In [ ]:
xgb_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('feature_selector', feature_selector),
    #('smote', smote),
    ('model', XGBClassifier(
        random_state=42
    ))
])

xgb_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_xgb = xgb_pipeline.predict(X_test)
print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))

#### **GridSearchCV for XGBoost**

In [ ]:
# Define parameter grid for XGBoost
param_grid_xgb = {
    'model__n_estimators': [100, 200, 300],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__max_depth': [3, 5, 7],
}

# Create GridSearchCV object
grid_search_xgb = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid_xgb,
    scoring='recall',
    cv=5,
    verbose=1,
    n_jobs=-1
)

# Fit GridSearchCV
grid_search_xgb.fit(X_train, y_train)

print("Best parameters for XGBoost (GridSearchCV):")
print(grid_search_xgb.best_params_)
print("Best recall score for XGBoost (GridSearchCV):")
print(grid_search_xgb.best_score_)

y_pred_xgb_grid = grid_search_xgb.predict(X_test)
print("\nXGBoost (GridSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_xgb_grid))

#### **RandomizedSearchCV for XGBoost**

In [ ]:
# Define parameter distributions for XGBoost
param_dist_xgb = {
    'model__n_estimators': randint(50, 500),
    'model__learning_rate': uniform(0.001, 0.3),
    'model__max_depth': randint(2, 10),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.6, 0.4),
    'model__gamma': uniform(0, 0.5),
    'model__reg_alpha': uniform(0, 1),  # L1 regularization on weights
    'model__reg_lambda': uniform(0, 1) # L2 regularization on weights
}

# Create RandomizedSearchCV object
random_search_xgb = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist_xgb,
    scoring='recall',
    n_iter=20, # Number of parameter settings that are sampled
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit RandomizedSearchCV
random_search_xgb.fit(X_train, y_train)

print("Best parameters for XGBoost (RandomizedSearchCV):")
print(random_search_xgb.best_params_)
print("Best recall score for XGBoost (RandomizedSearchCV):")
print(random_search_xgb.best_score_)

y_pred_xgb_rand = random_search_xgb.predict(X_test)
print("\nXGBoost (RandomizedSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_xgb_rand))

## **Model 5 - SVM**

In [ ]:
from sklearn.svm import SVC

svm_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('feature_selector', feature_selector),
    #('smote', smote),
    ('model', SVC(
        random_state=42
    ))
])

svm_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_svm = svm_pipeline.predict(X_test)
print("SVM Classification Report:")
print(classification_report(y_test, y_pred_svm))

#### **GridSearchCV for SVM**

In [ ]:
# Define parameter grid for SVM
param_grid_svm = {
    'model__C': [0.1, 1, 10],
    'model__gamma': ['scale', 'auto'],
    'model__kernel': ['rbf'] # Using 'rbf' for non-linear SVM
}

# Create GridSearchCV object
grid_search_svm = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid_svm,
    scoring='recall',
    cv=5,
    verbose=1,
    n_jobs=-1
)

# Fit GridSearchCV
grid_search_svm.fit(X_train, y_train)

print("Best parameters for SVM (GridSearchCV):")
print(grid_search_svm.best_params_)
print("Best recall score for SVM (GridSearchCV):")
print(grid_search_svm.best_score_)

y_pred_svm_grid = grid_search_svm.predict(X_test)
print("\nSVM (GridSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_svm_grid))

#### **RandomizedSearchCV for SVM**

In [ ]:
from scipy.stats import loguniform

# Define parameter distributions for SVM
param_dist_svm = {
    'model__C': loguniform(0.1, 100),
    'model__gamma': loguniform(0.001, 1),
    'model__kernel': ['rbf']
}

# Create RandomizedSearchCV object
random_search_svm = RandomizedSearchCV(
    estimator=svm_pipeline,
    param_distributions=param_dist_svm,
    scoring='recall',
    n_iter=20, # Number of parameter settings that are sampled
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit RandomizedSearchCV
random_search_svm.fit(X_train, y_train)

print("Best parameters for SVM (RandomizedSearchCV):")
print(random_search_svm.best_params_)
print("Best recall score for SVM (RandomizedSearchCV):")
print(random_search_svm.best_score_)

y_pred_svm_rand = random_search_svm.predict(X_test)
print("\nSVM (RandomizedSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_svm_rand))

## Model 6 - LightGBM

In [ ]:
from lightgbm import LGBMClassifier

lgbm_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('feature_selector', feature_selector),
   #('smote', smote),
    ('model', LGBMClassifier(
        random_state=42
    ))
])

lgbm_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_lgbm = lgbm_pipeline.predict(X_test)
print("LightGBM Classification Report:")
print(classification_report(y_test, y_pred_lgbm))

### GridSearchCV for LightGBM

In [ ]:
# Define parameter grid for LightGBM
param_grid_lgbm = {
    'model__n_estimators': [50, 100, 150],
    'model__learning_rate': [0.05, 0.1, 0.2],
    'model__num_leaves': [5, 20, 31],
    'model__max_depth': [-1, 5, 10]
}

# Create GridSearchCV object
grid_search_lgbm = GridSearchCV(
    estimator=lgbm_pipeline,
    param_grid=param_grid_lgbm,
    scoring='recall',
    cv=5,
    verbose=1,
    n_jobs=-1
)

# Fit GridSearchCV
grid_search_lgbm.fit(X_train, y_train)

print("Best parameters for LightGBM (GridSearchCV):")
print(grid_search_lgbm.best_params_)
print("Best recall score for LightGBM (GridSearchCV):")
print(grid_search_lgbm.best_score_)

y_pred_lgbm_grid = grid_search_lgbm.predict(X_test)
print("\nLightGBM (GridSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_lgbm_grid))

### RandomizedSearchCV for LightGBM

In [ ]:
# Define parameter distributions for LightGBM
param_dist_lgbm = {
    'model__n_estimators': randint(10, 200),
    'model__learning_rate': uniform(0, 1),
    'model__num_leaves': randint(5, 50),
    'model__max_depth': randint(-1, 10),
    'model__reg_alpha': uniform(0, 1), # L1 regularization
    'model__reg_lambda': uniform(0, 1) # L2 regularization
}

# Create RandomizedSearchCV object
random_search_lgbm = RandomizedSearchCV(
    estimator=lgbm_pipeline,
    param_distributions=param_dist_lgbm,
    scoring='recall',
    n_iter=20, # Number of parameter settings that are sampled
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit RandomizedSearchCV
random_search_lgbm.fit(X_train, y_train)

print("Best parameters for LightGBM (RandomizedSearchCV):")
print(random_search_lgbm.best_params_)
print("Best recall score for LightGBM (RandomizedSearchCV):")
print(random_search_lgbm.best_score_)

y_pred_lgbm_rand = random_search_lgbm.predict(X_test)
print("\nLightGBM (RandomizedSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_lgbm_rand))

## Model 7 - CatBoost

In [ ]:
cb_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('feature_selector', feature_selector),
    #('smote', smote),
    ('model', CatBoostClassifier(
        random_state=42
    ))
])

cb_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_cb = cb_pipeline.predict(X_test)
print("CatBoost Classification Report:")
print(classification_report(y_test, y_pred_cb))

### GridSearchCV for CatBoost

In [ ]:
# Define parameter grid for CatBoost
param_grid_cb = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3,4,5]
}

# Create GridSearchCV object
grid_search_cb = GridSearchCV(
    estimator=cb_pipeline,
    param_grid=param_grid_cb,
    scoring='recall',
    cv=5,
    verbose=1,
    n_jobs=-1
)

# Fit GridSearchCV
grid_search_cb.fit(X_train, y_train)

print("Best parameters for CatBoost (GridSearchCV):")
print(grid_search_cb.best_params_)
print("Best recall score for CatBoost (GridSearchCV):")
print(grid_search_cb.best_score_)

y_pred_cb_grid = grid_search_cb.predict(X_test)
print("\nCatBoost (GridSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_cb_grid))

### RandomizedSearchCV for CatBoost

In [ ]:
# Define parameter distributions for CatBoost
param_dist_cb = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth' : [3,4,5]
}

# Create RandomizedSearchCV object
random_search_cb = RandomizedSearchCV(
    estimator=cb_pipeline,
    param_distributions=param_dist_cb,
    scoring='recall',
    n_iter=20, # Number of parameter settings that are sampled
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit RandomizedSearchCV
random_search_cb.fit(X_train, y_train)

print("Best parameters for CatBoost (RandomizedSearchCV):")
print(random_search_cb.best_params_)
print("Best recall score for CatBoost (RandomizedSearchCV):")
print(random_search_cb.best_score_)

y_pred_cb_rand = random_search_cb.predict(X_test)
print("\nCatBoost (RandomizedSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_cb_rand))

## Model 8 - MLP

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('feature_selector', feature_selector),
    #('smote', smote),
    ('model', MLPClassifier(
        random_state=42
    ))
])

mlp_pipeline.fit(X_train, y_train)

In [ ]:
y_pred_mlp = mlp_pipeline.predict(X_test)
print("MLP Classification Report:")
print(classification_report(y_test, y_pred_mlp))

### GridSearchCV for MLP

In [ ]:
# Define parameter grid for MLP
param_grid_mlp = {
    'model__hidden_layer_sizes': [(10,30,10),(20,)],
    'model__activation': ['tanh', 'relu'],
    'model__solver': ['sgd', 'adam'],
    'model__alpha': [0.0001, 0.05],
    'model__learning_rate': ['constant','adaptive']
}

# Create GridSearchCV object
grid_search_mlp = GridSearchCV(
    estimator=mlp_pipeline,
    param_grid=param_grid_mlp,
    scoring='recall',
    cv=5,
    verbose=1,
    n_jobs=-1
)

# Fit GridSearchCV
grid_search_mlp.fit(X_train, y_train)

print("Best parameters for MLP (GridSearchCV):")
print(grid_search_mlp.best_params_)
print("Best recall score for MLP (GridSearchCV):")
print(grid_search_mlp.best_score_)

y_pred_mlp_grid = grid_search_mlp.predict(X_test)
print("\nMLP (GridSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_mlp_grid))

### RandomizedSearchCV for MLP

In [ ]:
from scipy.stats import uniform, randint

# Define parameter distributions for MLP
param_dist_mlp = {
    'model__hidden_layer_sizes': [(10,30,10),(20,)],
    'model__activation': ['tanh', 'relu'],
    'model__solver': ['sgd', 'adam'],
    'model__alpha': uniform(0.0001, 0.05),
    'model__learning_rate': ['constant','adaptive'],
    'model__learning_rate_init': uniform(0.001, 0.01)
}

# Create RandomizedSearchCV object
random_search_mlp = RandomizedSearchCV(
    estimator=mlp_pipeline,
    param_distributions=param_dist_mlp,
    scoring='recall',
    n_iter=20, # Number of parameter settings that are sampled
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit RandomizedSearchCV
random_search_mlp.fit(X_train, y_train)

print("Best parameters for MLP (RandomizedSearchCV):")
print(random_search_mlp.best_params_)
print("Best recall score for MLP (RandomizedSearchCV):")
print(random_search_mlp.best_score_)

y_pred_mlp_rand = random_search_mlp.predict(X_test)
print("\nMLP (RandomizedSearchCV) Classification Report:")
print(classification_report(y_test, y_pred_mlp_rand))

### **MLP Default Tuned (t=0.40)**

In [ ]:
y_proba_mlp_tuned_def = mlp_pipeline.predict_proba(X_test)[:, 1]
y_pred_mlp_tuned_def = (y_proba_mlp_tuned_def >= 0.40).astype(int)

print("MLP Default (t=0.40) — Final Model")
print(classification_report(y_test, y_pred_mlp_tuned_def))

### **MLP GridSearchCV Tuned (t=0.40)**

In [ ]:
y_proba_mlp_tuned_gr = grid_search_mlp.best_estimator_.predict_proba(X_test)[:, 1]
y_pred_mlp_tuned_gr = (y_proba_mlp_tuned_gr >= 0.40).astype(int)

print("MLP GridSearchCV (t=0.40) — Final Model")
print(classification_report(y_test, y_pred_mlp_tuned_gr))

### **MLP RandomizedSearchCV Tuned (t=0.40)**

In [ ]:
y_proba_mlp_tuned_rd = random_search_mlp.best_estimator_.predict_proba(X_test)[:, 1]
y_pred_mlp_tuned_rd = (y_proba_mlp_tuned_rd >= 0.40).astype(int)

print("MLP RandomizedSearchCV (t=0.40) — Final Model")
print(classification_report(y_test, y_pred_mlp_tuned_rd))

## **Model 9 - Stacking Classifier (Logistic Regression + MLP)**

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, roc_auc_score, precision_score, recall_score, f1_score

# Function to evaluate a model and plot its confusion matrix
def evaluate_and_plot_model(model, X_train, y_train, X_test, y_test, model_name):
    # Test Metrics
    y_pred_test = model.predict(X_test)
    y_proba_test = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else [0] * len(y_test) # For ROC AUC

    test_precision = precision_score(y_test, y_pred_test, zero_division=0)
    test_recall = recall_score(y_test, y_pred_test, zero_division=0)
    test_f1 = f1_score(y_test, y_pred_test, zero_division=0)
    test_roc_auc = roc_auc_score(y_test, y_proba_test) if hasattr(model, 'predict_proba') else None

    # Train Metrics
    y_pred_train = model.predict(X_train)
    y_proba_train = model.predict_proba(X_train)[:, 1] if hasattr(model, 'predict_proba') else [0] * len(y_train)

    train_precision = precision_score(y_train, y_pred_train, zero_division=0)
    train_recall = recall_score(y_train, y_pred_train, zero_division=0)
    train_f1 = f1_score(y_train, y_pred_train, zero_division=0)
    train_roc_auc = roc_auc_score(y_train, y_proba_train) if hasattr(model, 'predict_proba') else None


    print(f"--- {model_name} ---")
    print(f"Test Precision: {test_precision:.3f}")
    print(f"Test Recall: {test_recall:.3f}")
    print(f"Test F1-Score: {test_f1:.3f}")
    if hasattr(model, 'predict_proba'):
        print(f"Test ROC AUC: {test_roc_auc:.3f}")
    else:
        print("Test ROC AUC: Not available (model has no predict_proba)")

    print(f"Train Precision: {train_precision:.3f}")
    print(f"Train Recall: {train_recall:.3f}")
    print(f"Train F1-Score: {train_f1:.3f}")
    if hasattr(model, 'predict_proba'):
        print(f"Train ROC AUC: {train_roc_auc:.3f}")
    else:
        print("Train ROC AUC: Not available (model has no predict_proba)")


    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, cmap='Blues', ax=ax)
    ax.set_title(f'Confusion Matrix - {model_name}')
    plt.tight_layout()
    plt.show()

    # Return metrics for summary table
    return {
        'Model': model_name,
        'Test Precision': test_precision,
        'Test Recall': test_recall,
        'Test F1-Score': test_f1,
        'Test ROC AUC': test_roc_auc,
        'Train Precision': train_precision,
        'Train Recall': train_recall,
        'Train F1-Score': train_f1,
        'Train ROC AUC': train_roc_auc,
    }

# Helper function to evaluate model at a custom threshold and plot confusion matrix
def evaluate_and_plot_model_with_custom_threshold(model, X_train, y_train, X_test, y_test, model_name, threshold):
    # Test Metrics
    y_proba_test = model.predict_proba(X_test)[:, 1]
    y_pred_test = (y_proba_test >= threshold).astype(int)

    test_precision = precision_score(y_test, y_pred_test, zero_division=0)
    test_recall = recall_score(y_test, y_pred_test, zero_division=0)
    test_f1 = f1_score(y_test, y_pred_test, zero_division=0)
    test_roc_auc = roc_auc_score(y_test, y_proba_test)

    # Train Metrics
    y_proba_train = model.predict_proba(X_train)[:, 1]
    y_pred_train = (y_proba_train >= threshold).astype(int)

    train_precision = precision_score(y_train, y_pred_train, zero_division=0)
    train_recall = recall_score(y_train, y_pred_train, zero_division=0)
    train_f1 = f1_score(y_train, y_pred_train, zero_division=0)
    train_roc_auc = roc_auc_score(y_train, y_proba_train)

    print(f"--- {model_name} (t={threshold:.2f}) ---")
    print(f"Test Precision: {test_precision:.3f}")
    print(f"Test Recall: {test_recall:.3f}")
    print(f"Test F1-Score: {test_f1:.3f}")
    print(f"Test ROC AUC: {test_roc_auc:.3f}")

    print(f"Train Precision: {train_precision:.3f}")
    print(f"Train Recall: {train_recall:.3f}")
    print(f"Train F1-Score: {train_f1:.3f}")
    print(f"Train ROC AUC: {train_roc_auc:.3f}")

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_test, cmap='Blues', ax=ax)
    ax.set_title(f'Confusion Matrix - {model_name} (t={threshold:.2f})')
    plt.tight_layout()
    plt.show()

    return {
        'Model': f'{model_name} (t={threshold:.2f})',
        'Test Precision': test_precision,
        'Test Recall': test_recall,
        'Test F1-Score': test_f1,
        'Test ROC AUC': test_roc_auc,
        'Train Precision': train_precision,
        'Train Recall': train_recall,
        'Train F1-Score': train_f1,
        'Train ROC AUC': train_roc_auc,
    }

model_metrics = []


# Default Models (without tuning)
model_metrics.append(evaluate_and_plot_model(lr_pipeline, X_train, y_train, X_test, y_test, 'Logistic Regression (Default)'))
model_metrics.append(evaluate_and_plot_model(dt_pipeline, X_train, y_train, X_test, y_test, 'Decision Tree (Default)'))
model_metrics.append(evaluate_and_plot_model(rf_pipeline, X_train, y_train, X_test, y_test, 'Random Forest (Default)'))
model_metrics.append(evaluate_and_plot_model(xgb_pipeline, X_train, y_train, X_test, y_test, 'XGBoost (Default)'))
model_metrics.append(evaluate_and_plot_model(svm_pipeline, X_train, y_train, X_test, y_test, 'SVM (Default)'))
model_metrics.append(evaluate_and_plot_model(lgbm_pipeline, X_train, y_train, X_test, y_test, 'LightGBM (Default)'))
model_metrics.append(evaluate_and_plot_model(cb_pipeline, X_train, y_train, X_test, y_test, 'CatBoost (Default)'))
model_metrics.append(evaluate_and_plot_model(mlp_pipeline, X_train, y_train, X_test, y_test, 'MLP (Default)'))

# Logistic Regression
model_metrics.append(evaluate_and_plot_model(grid_search_lr.best_estimator_, X_train, y_train, X_test, y_test, 'Logistic Regression (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_lr.best_estimator_, X_train, y_train, X_test, y_test, 'Logistic Regression (RandomizedSearchCV)'))

# Logistic Regression (t=0.40) custom threshold
model_metrics.append(evaluate_and_plot_model_with_custom_threshold(lr_pipeline, X_train, y_train, X_test, y_test, 'Logistic Regression', threshold=0.40))

# Decision Tree
model_metrics.append(evaluate_and_plot_model(grid_search_dt.best_estimator_, X_train, y_train, X_test, y_test, 'Decision Tree (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_dt.best_estimator_, X_train, y_train, X_test, y_test, 'Decision Tree (RandomizedSearchCV)'))

# Random Forest
model_metrics.append(evaluate_and_plot_model(grid_search_rf.best_estimator_, X_train, y_train, X_test, y_test, 'Random Forest (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_rf.best_estimator_, X_train, y_train, X_test, y_test, 'Random Forest (RandomizedSearchCV)'))

# XGBoost
model_metrics.append(evaluate_and_plot_model(grid_search_xgb.best_estimator_, X_train, y_train, X_test, y_test, 'XGBoost (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_xgb.best_estimator_, X_train, y_train, X_test, y_test, 'XGBoost (RandomizedSearchCV)'))

# SVM
model_metrics.append(evaluate_and_plot_model(grid_search_svm.best_estimator_, X_train, y_train, X_test, y_test, 'SVM (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_svm.best_estimator_, X_train, y_train, X_test, y_test, 'SVM (RandomizedSearchCV)'))

# LightGBM
model_metrics.append(evaluate_and_plot_model(grid_search_lgbm.best_estimator_, X_train, y_train, X_test, y_test, 'LightGBM (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_lgbm.best_estimator_, X_train, y_train, X_test, y_test, 'LightGBM (RandomizedSearchCV)'))

# CatBoost
model_metrics.append(evaluate_and_plot_model(grid_search_cb.best_estimator_, X_train, y_train, X_test, y_test, 'CatBoost (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_cb.best_estimator_, X_train, y_train, X_test, y_test, 'CatBoost (RandomizedSearchCV)'))

# MLP
model_metrics.append(evaluate_and_plot_model(grid_search_mlp.best_estimator_, X_train, y_train, X_test, y_test, 'MLP (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_mlp.best_estimator_, X_train, y_train, X_test, y_test, 'MLP (RandomizedSearchCV)'))

# MLP (t=0.40) custom threshold evaluation
model_metrics.append(evaluate_and_plot_model_with_custom_threshold(mlp_pipeline, X_train, y_train, X_test, y_test, 'MLP (Default)', threshold=0.40))
model_metrics.append(evaluate_and_plot_model_with_custom_threshold(grid_search_mlp.best_estimator_, X_train, y_train, X_test, y_test, 'MLP (GridSearchCV)', threshold=0.40))
model_metrics.append(evaluate_and_plot_model_with_custom_threshold(random_search_mlp.best_estimator_, X_train, y_train, X_test, y_test, 'MLP (RandomizedSearchCV)', threshold=0.40))

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

# Each of these pipelines already includes the preprocessor and feature_selector steps.
estimators = [
    ('lr_tuned', grid_search_lr.best_estimator_),
    ('mlp_tuned', grid_search_mlp.best_estimator_)
]

final_estimator = LogisticRegression(
    random_state=42,
    solver='liblinear' # 'liblinear' solver supports both L1 and L2 penalties
)

# The StackingClassifier will manage passing the raw X_train/X_test data to its
# internal base estimators, and they will handle their own preprocessing and feature selection.
stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=final_estimator,
    cv=5,
    stack_method='predict_proba',
    n_jobs=-1,
    passthrough=False # The final estimator will learn from the meta-features (predictions of base models) only
)

# Fit the stacking model on the training data
stacking_model.fit(X_train, y_train)

# Get probability predictions for the test set
y_proba_stacking = stacking_model.predict_proba(X_test)[:, 1]

# Apply the 0.40 threshold to get binary predictions
y_pred_stacking_tuned = (y_proba_stacking >= 0.40).astype(int)

print("Stacking Classifier (LR + MLP, t=0.40) Classification Report:")
print(classification_report(y_test, y_pred_stacking_tuned))

# Display the confusion matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_stacking_tuned, cmap='Blues')
plt.title('Stacking Classifier (LR + MLP, t=0.40)')
plt.show()

# Add the evaluation metrics for the stacking model to the model_metrics list
metrics_entry_stacking = evaluate_and_plot_model_with_custom_threshold(
    stacking_model,
    X_train,
    y_train,
    X_test,
    y_test,
    'Stacking Classifier (LR + MLP)',
    threshold=0.40
)
model_metrics.append(metrics_entry_stacking)

## **Model Tuning Summary**

In [ ]:
tuned_models_summary = []

# Logistic Regression
tuned_models_summary.append({
    'Model': 'Logistic Regression',
    'Tuner': 'GridSearchCV',
    'Best Parameters': grid_search_lr.best_params_,
    'Best Recall Score': grid_search_lr.best_score_
})
tuned_models_summary.append({
    'Model': 'Logistic Regression',
    'Tuner': 'RandomizedSearchCV',
    'Best Parameters': random_search_lr.best_params_,
    'Best Recall Score': random_search_lr.best_score_
})

# Decision Tree
tuned_models_summary.append({
    'Model': 'Decision Tree',
    'Tuner': 'GridSearchCV',
    'Best Parameters': grid_search_dt.best_params_,
    'Best Recall Score': grid_search_dt.best_score_
})
tuned_models_summary.append({
    'Model': 'Decision Tree',
    'Tuner': 'RandomizedSearchCV',
    'Best Parameters': random_search_dt.best_params_,
    'Best Recall Score': random_search_dt.best_score_
})

# Random Forest
tuned_models_summary.append({
    'Model': 'Random Forest',
    'Tuner': 'GridSearchCV',
    'Best Parameters': grid_search_rf.best_params_,
    'Best Recall Score': grid_search_rf.best_score_
})
tuned_models_summary.append({
    'Model': 'Random Forest',
    'Tuner': 'RandomizedSearchCV',
    'Best Parameters': random_search_rf.best_params_,
    'Best Recall Score': random_search_rf.best_score_
})

# XGBoost
tuned_models_summary.append({
    'Model': 'XGBoost',
    'Tuner': 'GridSearchCV',
    'Best Parameters': grid_search_xgb.best_params_,
    'Best Recall Score': grid_search_xgb.best_score_
})
tuned_models_summary.append({
    'Model': 'XGBoost',
    'Tuner': 'RandomizedSearchCV',
    'Best Parameters': random_search_xgb.best_params_,
    'Best Recall Score': random_search_xgb.best_score_
})

# SVM
tuned_models_summary.append({
    'Model': 'SVM',
    'Tuner': 'GridSearchCV',
    'Best Parameters': grid_search_svm.best_params_,
    'Best Recall Score': grid_search_svm.best_score_
})
tuned_models_summary.append({
    'Model': 'SVM',
    'Tuner': 'RandomizedSearchCV',
    'Best Parameters': random_search_svm.best_params_,
    'Best Recall Score': random_search_svm.best_score_
})

# LightGBM
tuned_models_summary.append({
    'Model': 'LightGBM',
    'Tuner': 'GridSearchCV',
    'Best Parameters': grid_search_lgbm.best_params_,
    'Best Recall Score': grid_search_lgbm.best_score_
})
tuned_models_summary.append({
    'Model': 'LightGBM',
    'Tuner': 'RandomizedSearchCV',
    'Best Parameters': random_search_lgbm.best_params_,
    'Best Recall Score': random_search_lgbm.best_score_
})

# CatBoost
tuned_models_summary.append({
    'Model': 'CatBoost',
    'Tuner': 'GridSearchCV',
    'Best Parameters': grid_search_cb.best_params_,
    'Best Recall Score': grid_search_cb.best_score_
})
tuned_models_summary.append({
    'Model': 'CatBoost',
    'Tuner': 'RandomizedSearchCV',
    'Best Parameters': random_search_cb.best_params_,
    'Best Recall Score': random_search_cb.best_score_
})

# MLP
tuned_models_summary.append({
    'Model': 'MLP',
    'Tuner': 'GridSearchCV',
    'Best Parameters': grid_search_mlp.best_params_,
    'Best Recall Score': grid_search_mlp.best_score_
})
tuned_models_summary.append({
    'Model': 'MLP',
    'Tuner': 'RandomizedSearchCV',
    'Best Parameters': random_search_mlp.best_params_,
    'Best Recall Score': random_search_mlp.best_score_
})

# Stacking Classifier
# Assuming metrics_entry_stacking is available from the previous cell's execution
if 'metrics_entry_stacking' in locals() and metrics_entry_stacking is not None:
    tuned_models_summary.append({
        'Model': 'Stacking Classifier (LR + MLP) (t=0.40)',
        'Tuner': 'N/A', # Not directly tuned via GridSearchCV/RandomizedSearchCV
        'Best Parameters': 'N/A', # Base estimators are tuned, not the stacking ensemble directly
        'Best Recall Score': metrics_entry_stacking['Test Recall'] # Using Test Recall as the key metric
    })

summary_df = pd.DataFrame(tuned_models_summary)
display(summary_df.sort_values(by='Best Recall Score', ascending=False))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, roc_auc_score, precision_score, recall_score, f1_score

# Function to evaluate a model and plot its confusion matrix
def evaluate_and_plot_model(model, X_train, y_train, X_test, y_test, model_name):
    # Test Metrics
    y_pred_test = model.predict(X_test)
    y_proba_test = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else [0] * len(y_test) # For ROC AUC

    test_precision = precision_score(y_test, y_pred_test)
    test_recall = recall_score(y_test, y_pred_test)
    test_f1 = f1_score(y_test, y_pred_test)
    test_roc_auc = roc_auc_score(y_test, y_proba_test) if hasattr(model, 'predict_proba') else None

    # Train Metrics
    y_pred_train = model.predict(X_train)
    y_proba_train = model.predict_proba(X_train)[:, 1] if hasattr(model, 'predict_proba') else [0] * len(y_train)

    train_precision = precision_score(y_train, y_pred_train)
    train_recall = recall_score(y_train, y_pred_train)
    train_f1 = f1_score(y_train, y_pred_train)
    train_roc_auc = roc_auc_score(y_train, y_proba_train) if hasattr(model, 'predict_proba') else None


    print(f"--- {model_name} ---")
    print(f"Test Precision: {test_precision:.3f}")
    print(f"Test Recall: {test_recall:.3f}")
    print(f"Test F1-Score: {test_f1:.3f}")
    if hasattr(model, 'predict_proba'):
        print(f"Test ROC AUC: {test_roc_auc:.3f}")
    else:
        print("Test ROC AUC: Not available (model has no predict_proba)")

    print(f"Train Precision: {train_precision:.3f}")
    print(f"Train Recall: {train_recall:.3f}")
    print(f"Train F1-Score: {train_f1:.3f}")
    if hasattr(model, 'predict_proba'):
        print(f"Train ROC AUC: {train_roc_auc:.3f}")
    else:
        print("Train ROC AUC: Not available (model has no predict_proba)")


    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, cmap='Blues', ax=ax)
    ax.set_title(f'Confusion Matrix - {model_name}')
    plt.tight_layout()
    plt.show()

    # Return metrics for summary table
    return {
        'Model': model_name,
        'Test Precision': test_precision,
        'Test Recall': test_recall,
        'Test F1-Score': test_f1,
        'Test ROC AUC': test_roc_auc,
        'Train Precision': train_precision,
        'Train Recall': train_recall,
        'Train F1-Score': train_f1,
        'Train ROC AUC': train_roc_auc,
    }

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, roc_auc_score, precision_score, recall_score, f1_score

# Function to evaluate a model and plot its confusion matrix
def evaluate_and_plot_model(model, X_train, y_train, X_test, y_test, model_name):
    # Test Metrics
    y_pred_test = model.predict(X_test)
    y_proba_test = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else [0] * len(y_test) # For ROC AUC

    test_precision = precision_score(y_test, y_pred_test, zero_division=0)
    test_recall = recall_score(y_test, y_pred_test, zero_division=0)
    test_f1 = f1_score(y_test, y_pred_test, zero_division=0)
    test_roc_auc = roc_auc_score(y_test, y_proba_test) if hasattr(model, 'predict_proba') else None

    # Train Metrics
    y_pred_train = model.predict(X_train)
    y_proba_train = model.predict_proba(X_train)[:, 1] if hasattr(model, 'predict_proba') else [0] * len(y_train)

    train_precision = precision_score(y_train, y_pred_train, zero_division=0)
    train_recall = recall_score(y_train, y_pred_train, zero_division=0)
    train_f1 = f1_score(y_train, y_pred_train, zero_division=0)
    train_roc_auc = roc_auc_score(y_train, y_proba_train) if hasattr(model, 'predict_proba') else None


    print(f"--- {model_name} ---")
    print(f"Test Precision: {test_precision:.3f}")
    print(f"Test Recall: {test_recall:.3f}")
    print(f"Test F1-Score: {test_f1:.3f}")
    if hasattr(model, 'predict_proba'):
        print(f"Test ROC AUC: {test_roc_auc:.3f}")
    else:
        print("Test ROC AUC: Not available (model has no predict_proba)")

    print(f"Train Precision: {train_precision:.3f}")
    print(f"Train Recall: {train_recall:.3f}")
    print(f"Train F1-Score: {train_f1:.3f}")
    if hasattr(model, 'predict_proba'):
        print(f"Train ROC AUC: {train_roc_auc:.3f}")
    else:
        print("Train ROC AUC: Not available (model has no predict_proba)")


    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, cmap='Blues', ax=ax)
    ax.set_title(f'Confusion Matrix - {model_name}')
    plt.tight_layout()
    plt.show()

    # Return metrics for summary table
    return {
        'Model': model_name,
        'Test Precision': test_precision,
        'Test Recall': test_recall,
        'Test F1-Score': test_f1,
        'Test ROC AUC': test_roc_auc,
        'Train Precision': train_precision,
        'Train Recall': train_recall,
        'Train F1-Score': train_f1,
        'Train ROC AUC': train_roc_auc,
    }

# Helper function to evaluate model at a custom threshold and plot confusion matrix
def evaluate_and_plot_model_with_custom_threshold(model, X_train, y_train, X_test, y_test, model_name, threshold):
    # Test Metrics
    y_proba_test = model.predict_proba(X_test)[:, 1]
    y_pred_test = (y_proba_test >= threshold).astype(int)

    test_precision = precision_score(y_test, y_pred_test, zero_division=0)
    test_recall = recall_score(y_test, y_pred_test, zero_division=0)
    test_f1 = f1_score(y_test, y_pred_test, zero_division=0)
    test_roc_auc = roc_auc_score(y_test, y_proba_test)

    # Train Metrics
    y_proba_train = model.predict_proba(X_train)[:, 1]
    y_pred_train = (y_proba_train >= threshold).astype(int)

    train_precision = precision_score(y_train, y_pred_train, zero_division=0)
    train_recall = recall_score(y_train, y_pred_train, zero_division=0)
    train_f1 = f1_score(y_train, y_pred_train, zero_division=0)
    train_roc_auc = roc_auc_score(y_train, y_proba_train)

    print(f"--- {model_name} (t={threshold:.2f}) ---")
    print(f"Test Precision: {test_precision:.3f}")
    print(f"Test Recall: {test_recall:.3f}")
    print(f"Test F1-Score: {test_f1:.3f}")
    print(f"Test ROC AUC: {test_roc_auc:.3f}")

    print(f"Train Precision: {train_precision:.3f}")
    print(f"Train Recall: {train_recall:.3f}")
    print(f"Train F1-Score: {train_f1:.3f}")
    print(f"Train ROC AUC: {train_roc_auc:.3f}")

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_test, cmap='Blues', ax=ax)
    ax.set_title(f'Confusion Matrix - {model_name} (t={threshold:.2f})')
    plt.tight_layout()
    plt.show()

    return {
        'Model': f'{model_name} (t={threshold:.2f})',
        'Test Precision': test_precision,
        'Test Recall': test_recall,
        'Test F1-Score': test_f1,
        'Test ROC AUC': test_roc_auc,
        'Train Precision': train_precision,
        'Train Recall': train_recall,
        'Train F1-Score': train_f1,
        'Train ROC AUC': train_roc_auc,
    }

model_metrics = []


# Default Models (without tuning)
model_metrics.append(evaluate_and_plot_model(lr_pipeline, X_train, y_train, X_test, y_test, 'Logistic Regression (Default)'))
model_metrics.append(evaluate_and_plot_model(dt_pipeline, X_train, y_train, X_test, y_test, 'Decision Tree (Default)'))
model_metrics.append(evaluate_and_plot_model(rf_pipeline, X_train, y_train, X_test, y_test, 'Random Forest (Default)'))
model_metrics.append(evaluate_and_plot_model(xgb_pipeline, X_train, y_train, X_test, y_test, 'XGBoost (Default)'))
model_metrics.append(evaluate_and_plot_model(svm_pipeline, X_train, y_train, X_test, y_test, 'SVM (Default)'))
model_metrics.append(evaluate_and_plot_model(lgbm_pipeline, X_train, y_train, X_test, y_test, 'LightGBM (Default)'))
model_metrics.append(evaluate_and_plot_model(cb_pipeline, X_train, y_train, X_test, y_test, 'CatBoost (Default)'))
model_metrics.append(evaluate_and_plot_model(mlp_pipeline, X_train, y_train, X_test, y_test, 'MLP (Default)'))

# Logistic Regression
model_metrics.append(evaluate_and_plot_model(grid_search_lr.best_estimator_, X_train, y_train, X_test, y_test, 'Logistic Regression (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_lr.best_estimator_, X_train, y_train, X_test, y_test, 'Logistic Regression (RandomizedSearchCV)'))

# Logistic Regression (t=0.40) custom threshold
model_metrics.append(evaluate_and_plot_model_with_custom_threshold(lr_pipeline, X_train, y_train, X_test, y_test, 'Logistic Regression', threshold=0.40))

# Decision Tree
model_metrics.append(evaluate_and_plot_model(grid_search_dt.best_estimator_, X_train, y_train, X_test, y_test, 'Decision Tree (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_dt.best_estimator_, X_train, y_train, X_test, y_test, 'Decision Tree (RandomizedSearchCV)'))

# Random Forest
model_metrics.append(evaluate_and_plot_model(grid_search_rf.best_estimator_, X_train, y_train, X_test, y_test, 'Random Forest (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_rf.best_estimator_, X_train, y_train, X_test, y_test, 'Random Forest (RandomizedSearchCV)'))

# XGBoost
model_metrics.append(evaluate_and_plot_model(grid_search_xgb.best_estimator_, X_train, y_train, X_test, y_test, 'XGBoost (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_xgb.best_estimator_, X_train, y_train, X_test, y_test, 'XGBoost (RandomizedSearchCV)'))

# SVM
model_metrics.append(evaluate_and_plot_model(grid_search_svm.best_estimator_, X_train, y_train, X_test, y_test, 'SVM (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_svm.best_estimator_, X_train, y_train, X_test, y_test, 'SVM (RandomizedSearchCV)'))

# LightGBM
model_metrics.append(evaluate_and_plot_model(grid_search_lgbm.best_estimator_, X_train, y_train, X_test, y_test, 'LightGBM (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_lgbm.best_estimator_, X_train, y_train, X_test, y_test, 'LightGBM (RandomizedSearchCV)'))

# CatBoost
model_metrics.append(evaluate_and_plot_model(grid_search_cb.best_estimator_, X_train, y_train, X_test, y_test, 'CatBoost (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_cb.best_estimator_, X_train, y_train, X_test, y_test, 'CatBoost (RandomizedSearchCV)'))

# MLP
model_metrics.append(evaluate_and_plot_model(grid_search_mlp.best_estimator_, X_train, y_train, X_test, y_test, 'MLP (GridSearchCV)'))
model_metrics.append(evaluate_and_plot_model(random_search_mlp.best_estimator_, X_train, y_train, X_test, y_test, 'MLP (RandomizedSearchCV)'))

# MLP (t=0.40) custom threshold evaluation
model_metrics.append(evaluate_and_plot_model_with_custom_threshold(mlp_pipeline, X_train, y_train, X_test, y_test, 'MLP (Default)', threshold=0.40))
model_metrics.append(evaluate_and_plot_model_with_custom_threshold(grid_search_mlp.best_estimator_, X_train, y_train, X_test, y_test, 'MLP (GridSearchCV)', threshold=0.40))
model_metrics.append(evaluate_and_plot_model_with_custom_threshold(random_search_mlp.best_estimator_, X_train, y_train, X_test, y_test, 'MLP (RandomizedSearchCV)', threshold=0.40))

In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import recall_score

def plot_custom_threshold_learning_curve(estimator, X, y, threshold, title, axes=None, ylim=None, cv=None,
                                       n_jobs=None, train_sizes=np.linspace(.1, 1.0, 5)):
    if axes is None:
        _, axes = plt.subplots(1, 1, figsize=(10, 5))

    axes.set_title(title)
    if ylim is not None:
        axes.set_ylim(*ylim)
    axes.set_xlabel("Training examples")
    axes.set_ylabel("Recall Score")

    train_recalls = []
    test_recalls = []

    # Iterate over different training set sizes
    for train_size in train_sizes:
        train_indices = []
        test_indices = []

        # Use StratifiedKFold to generate indices for cross-validation
        from sklearn.model_selection import StratifiedKFold
        skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

        fold_train_recalls = []
        fold_test_recalls = []

        for train_idx, test_idx in skf.split(X, y):
            # Select a subset of the training data based on train_size
            num_train_samples = int(train_size * len(train_idx))
            if num_train_samples == 0:
                continue

            subset_train_idx = np.random.choice(train_idx, size=num_train_samples, replace=False)

            X_train_fold, X_test_fold = X.iloc[subset_train_idx], X.iloc[test_idx]
            y_train_fold, y_test_fold = y.iloc[subset_train_idx], y.iloc[test_idx]

            # Fit the estimator
            estimator.fit(X_train_fold, y_train_fold)

            # Predict probabilities and apply threshold for training set
            y_proba_train_fold = estimator.predict_proba(X_train_fold)[:, 1]
            y_pred_train_fold = (y_proba_train_fold >= threshold).astype(int)
            fold_train_recalls.append(recall_score(y_train_fold, y_pred_train_fold))

            # Predict probabilities and apply threshold for test set
            y_proba_test_fold = estimator.predict_proba(X_test_fold)[:, 1]
            y_pred_test_fold = (y_proba_test_fold >= threshold).astype(int)
            fold_test_recalls.append(recall_score(y_test_fold, y_pred_test_fold))

        if fold_train_recalls:
            train_recalls.append(np.mean(fold_train_recalls))
            test_recalls.append(np.mean(fold_test_recalls))

    train_recalls = np.array(train_recalls)
    test_recalls = np.array(test_recalls)
    train_sizes_actual = np.array([int(ts * len(X)) for ts in train_sizes])

    axes.grid()
    axes.plot(train_sizes_actual, train_recalls, "o-", color="r", label="Training recall")
    axes.plot(train_sizes_actual, test_recalls, "o-", color="g", label="Cross-validation recall")
    axes.legend(loc="best")

    return plt

### **Comparison of Model Metrics**

In [ ]:
metrics_df = pd.DataFrame(model_metrics)

# Ensure the Stacking Classifier metrics are in model_metrics before proceeding
# This addresses potential issues with model_metrics being reset or not having the stacking model added
stacking_model_name_full = 'Stacking Classifier (LR + MLP) (t=0.40)'
if 'stacking_model' in globals() and stacking_model_name_full not in [m['Model'] for m in model_metrics]:
    # Recalculate metrics_entry_stacking if it's missing from model_metrics
    # This assumes evaluate_and_plot_model_with_custom_threshold and other necessary variables are available
    metrics_entry_stacking_recalc = evaluate_and_plot_model_with_custom_threshold(
        stacking_model,
        X_train,
        y_train,
        X_test,
        y_test,
        'Stacking Classifier (LR + MLP)',
        threshold=0.40
    )
    model_metrics.append(metrics_entry_stacking_recalc)
    metrics_df = pd.DataFrame(model_metrics) # Recreate metrics_df with the updated model_metrics


# Create a new DataFrame for the requested comparison
recall_comparison_data = []

# Define the specific models to include in the comparison
selected_model_names = [
    'Logistic Regression (Default)',
    'Logistic Regression (GridSearchCV)',
    'Logistic Regression (RandomizedSearchCV)',
    'Logistic Regression (t=0.40)',

    'Decision Tree (Default)',
    'Decision Tree (GridSearchCV)',
    'Decision Tree (RandomizedSearchCV)',

    'Random Forest (Default)',
    'Random Forest (GridSearchCV)',
    'Random Forest (RandomizedSearchCV)',

    'XGBoost (Default)',
    'XGBoost (GridSearchCV)',
    'XGBoost (RandomizedSearchCV)',

    'SVM (Default)',
    'SVM (GridSearchCV)',
    'SVM (RandomizedSearchCV)',

    'LightGBM (Default)',
    'LightGBM (GridSearchCV)',
    'LightGBM (RandomizedSearchCV)',

    'CatBoost (Default)',
    'CatBoost (GridSearchCV)',
    'CatBoost (RandomizedSearchCV)',

    'MLP (Default)',
    'MLP (GridSearchCV)',
    'MLP (RandomizedSearchCV)',
    'MLP (Default) (t=0.40)',
    'MLP (GridSearchCV) (t=0.40)',
    'MLP (RandomizedSearchCV) (t=0.40)',
    'Stacking Classifier (LR + MLP) (t=0.40)'
]

# Filter the metrics_df to include only the selected models
filtered_metrics_df = metrics_df[metrics_df['Model'].isin(selected_model_names)].copy()

# Ensure the order of models is consistent if desired, or proceed with filtered data
# For simplicity, we'll iterate through the filtered_metrics_df

for model_name in selected_model_names:
    entry = filtered_metrics_df[filtered_metrics_df['Model'] == model_name]

    if not entry.empty:
        train_recall = entry['Train Recall'].values[0]
        test_recall = entry['Test Recall'].values[0]

        row_data = {
            'Model': model_name,
            'Train Recall': train_recall,
            'Test Recall': test_recall
        }

        # Calculate the recall gap
        if train_recall is not None and test_recall is not None:
            row_data['Recall Gap'] = train_recall - test_recall
        else:
            row_data['Recall Gap'] = None

        recall_comparison_data.append(row_data)

recall_comparison_df = pd.DataFrame(recall_comparison_data)

# Display the table, formatted to 3 decimal places
display(recall_comparison_df.round(3).fillna('N/A').sort_values(by='Test Recall', ascending=False))

In [ ]:
desired_order = [
    'Logistic Regression (Default)',
    'Logistic Regression (t=0.40)',
    'Logistic Regression (GridSearchCV)',
    'Logistic Regression (RandomizedSearchCV)',

    'Decision Tree (Default)',
    'Decision Tree (GridSearchCV)',
    'Decision Tree (RandomizedSearchCV)',

    'Random Forest (Default)',
    'Random Forest (GridSearchCV)',
    'Random Forest (RandomizedSearchCV)',

    'XGBoost (Default)',
    'XGBoost (GridSearchCV)',
    'XGBoost (RandomizedSearchCV)',

    'SVM (Default)',
    'SVM (GridSearchCV)',
    'SVM (RandomizedSearchCV)',

    'LightGBM (Default)',
    'LightGBM (GridSearchCV)',
    'LightGBM (RandomizedSearchCV)',

    'CatBoost (Default)',
    'CatBoost (GridSearchCV)',
    'CatBoost (RandomizedSearchCV)',

    'MLP (Default)',
    'MLP (GridSearchCV)',
    'MLP (RandomizedSearchCV)',
    'MLP (Default) (t=0.40)',
    'MLP (GridSearchCV) (t=0.40)',
    'MLP (RandomizedSearchCV) (t=0.40)',
    'Stacking Classifier (LR + MLP) (t=0.40)'
]

# Filter the DataFrame to include only the models in the desired_order
sorted_recall_comparison_df = recall_comparison_df[recall_comparison_df['Model'].isin(desired_order)].copy()

# Fill NaN values in 'Train Recall', 'Test Recall', and 'Recall Gap' columns before setting categorical type
sorted_recall_comparison_df['Train Recall'] = sorted_recall_comparison_df['Train Recall'].fillna(0) # Or another appropriate default
sorted_recall_comparison_df['Test Recall'] = sorted_recall_comparison_df['Test Recall'].fillna(0)
sorted_recall_comparison_df['Recall Gap'] = sorted_recall_comparison_df['Recall Gap'].fillna(0)

# Convert the 'Model' column to a categorical type with the desired order
sorted_recall_comparison_df['Model'] = pd.Categorical(
    sorted_recall_comparison_df['Model'],
    categories=desired_order,
    ordered=True
)

# Sort the DataFrame by the categorical 'Model' column
sorted_recall_comparison_df = sorted_recall_comparison_df.sort_values('Model')

display(sorted_recall_comparison_df.round(3))

### **Visualizing Key Metrics**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(18, 12)) # Create a 2x2 grid for subplots
axes = axes.flatten()

metrics_to_plot = ['Test Recall', 'Test Precision', 'Test F1-Score', 'Test ROC AUC']
titles = ['Test Recall Comparison', 'Test Precision Comparison', 'Test F1-Score Comparison', 'Test ROC AUC Comparison']

unique_models = metrics_df['Model'].unique()

other_colors = sns.color_palette('tab10', len(unique_models) - 5)

custom_palette = {}
color_idx = 0
for model in unique_models:
    if model == 'Logistic Regression (t=0.40)':
        custom_palette[model] = 'red'
    elif model == 'MLP (Default) (t=0.40)':
        custom_palette[model] = 'purple'
    elif model == 'MLP (GridSearchCV) (t=0.40)':
        custom_palette[model] = 'darkviolet'
    elif model == 'MLP (RandomizedSearchCV) (t=0.40)':
        custom_palette[model] = 'magenta'
    elif model == 'Stacking Classifier (LR + MLP)':
        custom_palette[model] = 'cyan'
    else:
        custom_palette[model] = other_colors[color_idx % len(other_colors)] # Cycle through colors
        color_idx += 1

for i, metric in enumerate(metrics_to_plot):
    # Sort the dataframe by the current metric for better visualization
    sorted_df = metrics_df.sort_values(by=metric, ascending=False)

    # Map model names to colors for the current sorted order
    model_colors = [custom_palette.get(model, 'grey') for model in sorted_df['Model']]

    sns.barplot(x='Model', y=metric, data=sorted_df, ax=axes[i], palette=model_colors, hue='Model', legend=False)
    axes[i].set_title(titles[i])
    axes[i].set_ylabel(metric)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_ylim(0, 1) # Metrics are between 0 and 1

plt.suptitle('Comparison of Model Performance Metrics on Test Set', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## **Learning Curves**

In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np
import matplotlib.pyplot as plt

def plot_model_learning_curve(estimator, X, y, title, axes=None, ylim=None, cv=None,
                            n_jobs=None, train_sizes=np.linspace(.1, 1.0, 5)):
    if axes is None:
        _, axes = plt.subplots(1, 1, figsize=(10, 5))

    axes.set_title(title)
    if ylim is not None:
        axes.set_ylim(*ylim)
    axes.set_xlabel("Training examples")
    axes.set_ylabel("Recall Score")

    train_sizes, train_scores, test_scores = learning_curve(
        estimator,
        X,
        y,
        cv=cv,
        n_jobs=n_jobs,
        train_sizes=train_sizes,
        scoring='recall',
        error_score="raise"
    )
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)

    # Plot learning curve
    axes.grid()
    axes.fill_between(
        train_sizes,
        train_scores_mean - train_scores_std,
        train_scores_mean + train_scores_std,
        alpha=0.1,
        color="r",
    )
    axes.fill_between(
        train_sizes,
        test_scores_mean - test_scores_std,
        test_scores_mean + test_scores_std,
        alpha=0.1,
        color="g",
    )
    axes.plot(
        train_sizes, train_scores_mean, "o-", color="r", label="Training recall"
    )
    axes.plot(
        train_sizes, test_scores_mean, "o-", color="g", label="Cross-validation recall"
    )
    axes.legend(loc="best")

    return plt

In [ ]:
# Plotting Learning Curves for Logistic Regression
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)
plot_model_learning_curve(lr_pipeline, X_train, y_train, "Logistic Regression (Default)", axes=axes[0], cv=5, n_jobs=-1, ylim=(0.5, 1.0))
plot_model_learning_curve(grid_search_lr.best_estimator_, X_train, y_train, "Logistic Regression (GridSearchCV)", axes=axes[1], cv=5, n_jobs=-1, ylim=(0.5, 1.0))
plot_model_learning_curve(random_search_lr.best_estimator_, X_train, y_train, "Logistic Regression (RandomizedSearchCV)", axes=axes[2], cv=5, n_jobs=-1, ylim=(0.5, 1.0))
plt.suptitle("Learning Curves for Logistic Regression Models", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Plotting Learning Curves for Decision Tree
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)
plot_model_learning_curve(dt_pipeline, X_train, y_train, "Decision Tree (Default)", axes=axes[0], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plot_model_learning_curve(grid_search_dt.best_estimator_, X_train, y_train, "Decision Tree (GridSearchCV)", axes=axes[1], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plot_model_learning_curve(random_search_dt.best_estimator_, X_train, y_train, "Decision Tree (RandomizedSearchCV)", axes=axes[2], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plt.suptitle("Learning Curves for Decision Tree Models", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Plotting Learning Curves for Random Forest
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)
plot_model_learning_curve(rf_pipeline, X_train, y_train, "Random Forest (Default)", axes=axes[0], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plot_model_learning_curve(grid_search_rf.best_estimator_, X_train, y_train, "Random Forest (GridSearchCV)", axes=axes[1], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plot_model_learning_curve(random_search_rf.best_estimator_, X_train, y_train, "Random Forest (RandomizedSearchCV)", axes=axes[2], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plt.suptitle("Learning Curves for Random Forest Models", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Plotting Learning Curves for XGBoost
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)
plot_model_learning_curve(xgb_pipeline, X_train, y_train, "XGBoost (Default)", axes=axes[0], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plot_model_learning_curve(grid_search_xgb.best_estimator_, X_train, y_train, "XGBoost (GridSearchCV)", axes=axes[1], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plot_model_learning_curve(random_search_xgb.best_estimator_, X_train, y_train, "XGBoost (RandomizedSearchCV)", axes=axes[2], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plt.suptitle("Learning Curves for XGBoost Models", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Plotting Learning Curves for SVM
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)
plot_model_learning_curve(svm_pipeline, X_train, y_train, "SVM (Default)", axes=axes[0], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plot_model_learning_curve(grid_search_svm.best_estimator_, X_train, y_train, "SVM (GridSearchCV)", axes=axes[1], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plot_model_learning_curve(random_search_svm.best_estimator_, X_train, y_train, "SVM (RandomizedSearchCV)", axes=axes[2], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plt.suptitle("Learning Curves for SVM Models", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Plotting Learning Curves for LightGBM
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)
plot_model_learning_curve(lgbm_pipeline, X_train, y_train, "LightGBM (Default)", axes=axes[0], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plot_model_learning_curve(grid_search_lgbm.best_estimator_, X_train, y_train, "LightGBM (GridSearchCV)", axes=axes[1], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plot_model_learning_curve(random_search_lgbm.best_estimator_, X_train, y_train, "LightGBM (RandomizedSearchCV)", axes=axes[2], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plt.suptitle("Learning Curves for LightGBM Models", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Plotting Learning Curves for CatBoost
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)
plot_model_learning_curve(cb_pipeline, X_train, y_train, "CatBoost (Default)", axes=axes[0], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plot_model_learning_curve(grid_search_cb.best_estimator_, X_train, y_train, "CatBoost (GridSearchCV)", axes=axes[1], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plot_model_learning_curve(random_search_cb.best_estimator_, X_train, y_train, "CatBoost (RandomizedSearchCV)", axes=axes[2], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plt.suptitle("Learning Curves for CatBoost Models", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Plotting Learning Curves for MLP
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)
plot_model_learning_curve(mlp_pipeline, X_train, y_train, "MLP (Default)", axes=axes[0], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plot_model_learning_curve(grid_search_mlp.best_estimator_, X_train, y_train, "MLP (GridSearchCV)", axes=axes[1], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plot_model_learning_curve(random_search_mlp.best_estimator_, X_train, y_train, "MLP (RandomizedSearchCV)", axes=axes[2], cv=5, n_jobs=-1, ylim=(0.4, 1.0))
plt.suptitle("Learning Curves for MLP Models", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import recall_score

def plot_custom_threshold_learning_curve(estimator, X, y, threshold, title, axes=None, ylim=None, cv=None,
                                       n_jobs=None, train_sizes=np.linspace(.1, 1.0, 5)):
    if axes is None:
        _, axes = plt.subplots(1, 1, figsize=(10, 5))

    axes.set_title(title)
    if ylim is not None:
        axes.set_ylim(*ylim)
    axes.set_xlabel("Training examples")
    axes.set_ylabel("Recall Score")

    train_recalls = []
    test_recalls = []

    # Iterate over different training set sizes
    for train_size in train_sizes:
        train_indices = []
        test_indices = []

        # Use StratifiedKFold to generate indices for cross-validation
        from sklearn.model_selection import StratifiedKFold
        skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)

        fold_train_recalls = []
        fold_test_recalls = []

        for train_idx, test_idx in skf.split(X, y):
            # Select a subset of the training data based on train_size
            num_train_samples = int(train_size * len(train_idx))
            if num_train_samples == 0:
                continue

            subset_train_idx = np.random.choice(train_idx, size=num_train_samples, replace=False)

            X_train_fold, X_test_fold = X.iloc[subset_train_idx], X.iloc[test_idx]
            y_train_fold, y_test_fold = y.iloc[subset_train_idx], y.iloc[test_idx]

            # Fit the estimator
            estimator.fit(X_train_fold, y_train_fold)

            # Predict probabilities and apply threshold for training set
            y_proba_train_fold = estimator.predict_proba(X_train_fold)[:, 1]
            y_pred_train_fold = (y_proba_train_fold >= threshold).astype(int)
            fold_train_recalls.append(recall_score(y_train_fold, y_pred_train_fold))

            # Predict probabilities and apply threshold for test set
            y_proba_test_fold = estimator.predict_proba(X_test_fold)[:, 1]
            y_pred_test_fold = (y_proba_test_fold >= threshold).astype(int)
            fold_test_recalls.append(recall_score(y_test_fold, y_pred_test_fold))

        if fold_train_recalls:
            train_recalls.append(np.mean(fold_train_recalls))
            test_recalls.append(np.mean(fold_test_recalls))

    train_recalls = np.array(train_recalls)
    test_recalls = np.array(test_recalls)
    train_sizes_actual = np.array([int(ts * len(X)) for ts in train_sizes])

    axes.grid()
    axes.plot(train_sizes_actual, train_recalls, "o-", color="r", label="Training recall")
    axes.plot(train_sizes_actual, test_recalls, "o-", color="g", label="Cross-validation recall")
    axes.legend(loc="best")

    return plt

# Create a new figure and axes for the custom threshold learning curve
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot the learning curve for Logistic Regression with t=0.40
plot_custom_threshold_learning_curve(
    lr_pipeline,
    X_train,
    y_train,
    threshold=0.40,
    title="Logistic Regression (t=0.40) Learning Curve",
    axes=ax,
    cv=5,
    n_jobs=-1,
    ylim=(0.5, 1.0)
)

plt.suptitle("Learning Curve for Logistic Regression (t=0.40)", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)

plot_custom_threshold_learning_curve(
    mlp_pipeline,
    X_train,
    y_train,
    threshold=0.40,
    title="MLP (Default, t=0.40) Learning Curve",
    axes=axes[0],
    cv=5,
    n_jobs=-1,
    ylim=(0.5, 1.0)
)

plot_custom_threshold_learning_curve(
    grid_search_mlp.best_estimator_,
    X_train,
    y_train,
    threshold=0.40,
    title="MLP (GridSearchCV, t=0.40) Learning Curve",
    axes=axes[1],
    cv=5,
    n_jobs=-1,
    ylim=(0.5, 1.0)
)

plot_custom_threshold_learning_curve(
    random_search_mlp.best_estimator_,
    X_train,
    y_train,
    threshold=0.40,
    title="MLP (RandomizedSearchCV, t=0.40) Learning Curve",
    axes=axes[2],
    cv=5,
    n_jobs=-1,
    ylim=(0.5, 1.0)
)

plt.suptitle("Learning Curves for MLP Models (t=0.40)", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# Plotting Learning Curve for Stacking Classifier with t=0.40
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

plot_custom_threshold_learning_curve(
    stacking_model,
    X_train,
    y_train,
    threshold=0.40,
    title="Stacking Classifier (LR + MLP, t=0.40) Learning Curve",
    axes=ax,
    cv=5,
    n_jobs=-1,
    ylim=(0.5, 1.0)
)

plt.suptitle("Learning Curve for Stacking Classifier (LR + MLP, t=0.40)", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## **Bias & Fairness Assessment**

### **Categorizing Age**

In [ ]:
import numpy as np

# Divide age into bins and labels
age_bins = [18, 25, 35, 45, 55, 65]
age_labels = ['18-24', '25-34', '35-44', '45-54', '55-64']

df_fairness = df.copy()
df_fairness['age_group'] = pd.cut(df_fairness['age'], bins=age_bins, labels=age_labels, right=False)

display(df_fairness['age_group'].value_counts().sort_index())

### **Churn Distribution by Protected Attributes**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Churn by Gender
sns.countplot(data=df_fairness, x='gender', hue='churn', palette='viridis', ax=axes[0])
axes[0].set_title('Churn Distribution by Gender')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Count')
axes[0].legend(title='Churn', labels=['Stayed (0)', 'Churned (1)'])

# Churn by Age Group
sns.countplot(data=df_fairness, x='age_group', hue='churn', palette='magma', ax=axes[1])
axes[1].set_title('Churn Distribution by Age Group')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Churn', labels=['Stayed (0)', 'Churned (1)'])

plt.tight_layout()
plt.show()

# Display churn rates for each group
print(f"Churn Rate by Gender:")
display(df_fairness.groupby('gender')['churn'].mean().reset_index().round(3))

print(f"Churn Rate by Age Group:")
display(df_fairness.groupby('age_group')['churn'].mean().reset_index().round(3))

### **Model Performance (Logistic Regression t=0.40) by Protected Attributes**

In [ ]:
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score

def evaluate_model_by_group(model, X_test_original, y_test_original, protected_attribute, threshold=0.5):
    results = []

    # Make a copy to add the predicted churn without modifying the original X_test
    # We need to ensure that the protected_attribute column exists in df_fairness for lookup
    if protected_attribute not in df_fairness.columns:
        print(f"Warning: '{protected_attribute}' not found in df_fairness. Skipping.")
        return pd.DataFrame()

    X_test_with_attr = X_test_original.copy()
    X_test_with_attr[protected_attribute] = df_fairness.loc[X_test_original.index, protected_attribute]

    for group_name in X_test_with_attr[protected_attribute].unique():
        group_mask = (X_test_with_attr[protected_attribute] == group_name)
        X_group = X_test_original[group_mask]
        y_group = y_test_original[group_mask]

        if len(y_group) == 0: # Skip if no samples in this group
            continue

        if hasattr(model, 'predict_proba'):
            y_proba = model.predict_proba(X_group)[:, 1]
            y_pred = (y_proba >= threshold).astype(int)
            roc_auc = roc_auc_score(y_group, y_proba) if len(y_group.unique()) > 1 else None # ROC AUC requires at least two classes
        else:
            y_pred = model.predict(X_group)
            y_proba = [0] * len(y_group) # Placeholder if no predict_proba
            roc_auc = None

        recall = recall_score(y_group, y_pred, zero_division=0)
        precision = precision_score(y_group, y_pred, zero_division=0)
        f1 = f1_score(y_group, y_pred, zero_division=0)

        results.append({
            protected_attribute: group_name,
            'Count': len(y_group),
            'Churned Actual': y_group.sum(),
            'Recall': recall,
            'Precision': precision,
            'F1-Score': f1,
            'ROC AUC': roc_auc
        })
    return pd.DataFrame(results)

# Identify all categorical columns from the original df that are suitable for fairness assessment
# Exclude 'churn_period' as it's directly related to the target and dropped earlier
categorical_for_fairness = [col for col in df_fairness.select_dtypes(include='object').columns if col != 'churn_period']
categorical_for_fairness.append('age_group') # Add the newly created age group

# Evaluate by each categorical feature
for feature in categorical_for_fairness:
    print(f"\nPerformance by {feature} (Logistic Regression t=0.40):")
    results = evaluate_model_by_group(lr_pipeline, X_test, y_test, feature, threshold=0.40)
    display(results.round(3))

### **Model Performance (MLP GridSearchCV Tuned t=0.40) by Protected Attributes**

In [ ]:
for feature in categorical_for_fairness:
    print(f"\nPerformance by {feature} (MLP GridSearchCV t=0.40):")
    results = evaluate_model_by_group(grid_search_mlp.best_estimator_, X_test, y_test, feature, threshold=0.40)
    display(results.round(3))

# **Model Explainability Report (SHAP)**

In [ ]:
import shap

# Get feature names after preprocessing and feature selection

# 1. Transform X_train using the preprocessor to get processed feature names
X_train_processed = preprocessor.fit_transform(X_train)

# 2. Get all feature names after preprocessing
all_feature_names = preprocessor.get_feature_names_out()

# 3. Get the boolean mask of selected features from the feature_selector
# The feature_selector was already fitted in previous steps, but we can refit it on the processed data to be safe.
# Here we are using the feature_selector defined in cell LMtHEnRsVxmY
feature_selector.fit(X_train_processed, y_train)
selected_features_mask = feature_selector.get_support()

# 4. Get the names of the final selected features
final_feature_names = all_feature_names[selected_features_mask]

print("Final features used by the models:")
print(final_feature_names)

# Ensure the lr_pipeline is fitted, which it is from previous cells.

# Extract the fitted Logistic Regression model from the pipeline
lr_model = lr_pipeline.named_steps['model']

# Preprocess X_train and X_test using the pipeline's preprocessor and feature_selector
X_train_preprocessed_selected = lr_pipeline.named_steps['feature_selector'].transform(
    lr_pipeline.named_steps['preprocessor'].transform(X_train)
)
X_test_preprocessed_selected = lr_pipeline.named_steps['feature_selector'].transform(
    lr_pipeline.named_steps['preprocessor'].transform(X_test)
)

# Convert the processed arrays back to DataFrames with correct feature names
# 'final_feature_names' is available from cell '93255a6e'
X_train_final_df = pd.DataFrame(X_train_preprocessed_selected, columns=final_feature_names)
X_test_final_df = pd.DataFrame(X_test_preprocessed_selected, columns=final_feature_names)


# Create a LinearExplainer for the Logistic Regression model
# Pass the processed training data as the background dataset
explainer = shap.LinearExplainer(lr_model, X_train_final_df)

# Calculate SHAP values
shap_values = explainer.shap_values(X_test_final_df)

# Plot summary
shap.summary_plot(shap_values, X_test_final_df, show=False)
plt.suptitle('SHAP Summary Plot for Logistic Regression (Default)', x=0.5, y=1.02, ha='center')
plt.tight_layout()
plt.show()

In [ ]:
print("best_k:", best_k)
print("final_feature_names:", final_feature_names)
print("Jumlah fitur:", len(final_feature_names))

In [ ]:
import shap

# Ensure the lr_pipeline is fitted, which it is from previous cells.

# Extract the fitted Logistic Regression model from the pipeline
lr_model = lr_pipeline.named_steps['model']

# Preprocess X_train and X_test using the pipeline's preprocessor and feature_selector
# These steps are crucial to ensure that the data fed to SHAP is in the correct format
# expected by the model within the pipeline.
X_train_preprocessed = lr_pipeline.named_steps['preprocessor'].transform(X_train)
X_test_preprocessed = lr_pipeline.named_steps['preprocessor'].transform(X_test)

X_train_preprocessed_selected = lr_pipeline.named_steps['feature_selector'].transform(X_train_preprocessed)
X_test_preprocessed_selected = lr_pipeline.named_steps['feature_selector'].transform(X_test_preprocessed)

# Use the already defined 'final_feature_names' to create DataFrames
# 'final_feature_names' were determined in cell `93255a6e` or `w_4pKy3fYEZ7`.
X_train_final_df = pd.DataFrame(X_train_preprocessed_selected, columns=final_feature_names)
X_test_final_df = pd.DataFrame(X_test_preprocessed_selected, columns=final_feature_names)

# Create a LinearExplainer for the Logistic Regression model
# We pass the processed training data (X_train_final_df) as the background dataset
# for the explainer to estimate feature distributions.
explainer = shap.LinearExplainer(lr_model, X_train_final_df)

# Calculate SHAP values for the test set
shap_values = explainer.shap_values(X_test_final_df)

# Visualize the SHAP summary plot
# This plot shows the impact of each feature on the model output across the test dataset.
# The 'tuned threshold > 0.40' specifically refers to how the model's output (probabilities)
# are interpreted for classification, but SHAP itself explains the contribution to those probabilities.
shap.summary_plot(shap_values, X_test_final_df, show=False)
plt.suptitle('SHAP Summary Plot for Logistic Regression (Tuned with Threshold > 0.40)', x=0.5, y=1.02, ha='center')
plt.tight_layout()
plt.show()

In [ ]:
mean_shap = np.abs(shap_values).mean(axis=0)
for name, val in zip(final_feature_names, mean_shap):
    print(f"{name}: {val:.4f}")

## **Best Model Selection**

To determine the best-fit model, we need to consider a balanced metric that accounts for both false positives and false negatives, especially given the potential for overfitting when a model achieves perfect recall. The **F1-Score** is an excellent choice as it is the harmonic mean of precision and recall.

A high F1-Score indicates a good balance between identifying actual churn cases (recall) and minimizing incorrect churn predictions (precision).

After re-evaluating all models, including both default and tuned versions, the **Logistic Regression (t=0.40)**, **MLP (Default) (t=0.40)** and **MLP (GridSearchCV) (t=0.40)** models achieved the highest 'Test F1-Score', indicating a strong balance in their performance and the smallest 'Recall Gap', indicating the model is not overfit/underfit.

Here is a summary of the top-performing models based on 'Test F1-Score':

In [ ]:
# Sort the metrics DataFrame by 'Test F1-Score' to identify top performers
all_models_sorted_f1 = metrics_df.sort_values(by='Test F1-Score', ascending=False)

# Define the number of top models to show (adjust as needed)
num_top_models = 10

# Get the top N models initially
top_n_models = all_models_sorted_f1.head(num_top_models)[['Model', 'Test Precision', 'Test Recall', 'Test F1-Score', 'Test ROC AUC']]

# Find the row for 'Logistic Regression (t=0.40)'
lr_t_40_model_row = metrics_df[metrics_df['Model'] == 'Logistic Regression (t=0.40)'][['Model', 'Test Precision', 'Test Recall', 'Test F1-Score', 'Test ROC AUC']]

# Find the rows for MLP tuned models (t=0.40)
mlp_default_t_40_model_row = metrics_df[metrics_df['Model'] == 'MLP (Default) (t=0.40)'][['Model', 'Test Precision', 'Test Recall', 'Test F1-Score', 'Test ROC AUC']]
mlp_grid_t_40_model_row = metrics_df[metrics_df['Model'] == 'MLP (GridSearchCV) (t=0.40)'][['Model', 'Test Precision', 'Test Recall', 'Test F1-Score', 'Test ROC AUC']]
mlp_rand_t_40_model_row = metrics_df[metrics_df['Model'] == 'MLP (RandomizedSearchCV) (t=0.40)'][['Model', 'Test Precision', 'Test Recall', 'Test F1-Score', 'Test ROC AUC']]

# Start with the top N models
models_to_display = top_n_models

# Add 'Logistic Regression (t=0.40)' if not already in top N
if 'Logistic Regression (t=0.40)' not in models_to_display['Model'].values and not lr_t_40_model_row.empty:
    models_to_display = pd.concat([models_to_display, lr_t_40_model_row], ignore_index=True)

# Add 'MLP (Default) (t=0.40)' if not already in top N
if 'MLP (Default) (t=0.40)' not in models_to_display['Model'].values and not mlp_default_t_40_model_row.empty:
    models_to_display = pd.concat([models_to_display, mlp_default_t_40_model_row], ignore_index=True)

# Add 'MLP (GridSearchCV) (t=0.40)' if not already in top N
if 'MLP (GridSearchCV) (t=0.40)' not in models_to_display['Model'].values and not mlp_grid_t_40_model_row.empty:
    models_to_display = pd.concat([models_to_display, mlp_grid_t_40_model_row], ignore_index=True)

# Add 'MLP (RandomizedSearchCV) (t=0.40)' if not already in top N
if 'MLP (RandomizedSearchCV) (t=0.40)' not in models_to_display['Model'].values and not mlp_rand_t_40_model_row.empty:
    models_to_display = pd.concat([models_to_display, mlp_rand_t_40_model_row], ignore_index=True)

# Re-sort the combined DataFrame to ensure correct order after adding
models_to_display = models_to_display.sort_values(by='Test F1-Score', ascending=False)

# Display the resulting DataFrame, rounded to 3 decimal places
display(models_to_display.round(3))

The **Logistic Regression (t=0.40)**, **MLP (Default) (t=0.40)** and **MLP (GridSearchCV) (t=0.40)** models demonstrate a robust balance between precision and recall, as indicated by their top F1-scores. While some models previously showed perfect recall, their lower precision suggested overfitting. The Logistic Regression (t=0.40) and MLP (t=0.40) models, on the other hand, offer a more reliable performance by effectively identifying churners without generating an excessive number of false alarms.

## **Feature Importance Analysis**

In [ ]:
# Get feature names after preprocessing and feature selection

# 1. Transform X_train using the preprocessor to get processed feature names
X_train_processed = preprocessor.fit_transform(X_train)

# 2. Get all feature names after preprocessing
all_feature_names = preprocessor.get_feature_names_out()

# 3. Get the boolean mask of selected features from the feature_selector
# The feature_selector was already fitted in previous steps, but we can refit it on the processed data to be safe.
# Here we are using the feature_selector defined in cell LMtHEnRsVxmY
feature_selector.fit(X_train_processed, y_train)
selected_features_mask = feature_selector.get_support()

# 4. Get the names of the final selected features
final_feature_names = all_feature_names[selected_features_mask]

print("Final features used by the models:")
print(final_feature_names)

### **Logistic Regression Feature Coefficients**

In [ ]:
def plot_feature_importance(model, feature_names, model_name, coef=False):
    if coef:
        # For Logistic Regression, use coefficients
        # Access the model directly from the pipeline's 'model' step
        importances = model.named_steps['model'].coef_[0]
        title = f'Feature Coefficients - {model_name}'
        ylabel = 'Coefficient Value'
    else:
        # For tree-based models, use feature_importances_
        # Access the model directly from the pipeline's 'model' step
        # Check if the model has feature_importances_ attribute
        if hasattr(model.named_steps['model'], 'feature_importances_'):
            importances = model.named_steps['model'].feature_importances_
        else:
            print(f"Model {model_name} does not have feature_importances_ attribute.")
            return # Exit the function if feature_importances_ is not available

        title = f'Feature Importance - {model_name}'
        ylabel = 'Importance Score'

    feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
    plt.title(title)
    plt.xlabel(ylabel)
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

    print(f"\n--- {title} ---")
    display(feature_importance_df.head(10)) # Display top 10 for brevity

In [ ]:
plot_feature_importance(lr_pipeline, final_feature_names, 'Logistic Regression (Default)', coef=True)

In [ ]:
plot_feature_importance(lr_pipeline, final_feature_names, 'Logistic Regression (t=0.40)', coef=True)

In [ ]:
# Logistic Regression (GridSearchCV) - using best estimator
plot_feature_importance(grid_search_lr.best_estimator_, final_feature_names, 'Logistic Regression (GridSearchCV)', coef=True)

### **Decision Tree Feature Importance**

In [ ]:
# Logistic Regression (RandomizedSearchCV) - using best estimator
plot_feature_importance(random_search_lr.best_estimator_, final_feature_names, 'Logistic Regression (RandomizedSearchCV)', coef=True)

In [ ]:
# Decision Tree (GridSearchCV) - using best estimator
plot_feature_importance(grid_search_dt.best_estimator_, final_feature_names, 'Decision Tree (GridSearchCV')

### **Random Forest Feature Importance**

In [ ]:
# Random Forest (GridSearchCV) - using best estimator
plot_feature_importance(grid_search_rf.best_estimator_, final_feature_names, 'Random Forest (GridSearchCV)')

### **XGBoost Feature Importance**

In [ ]:
# XGBoost (GridSearchCV) - using best estimator
plot_feature_importance(grid_search_xgb.best_estimator_, final_feature_names, 'XGBoost (GridSearchCV)')

### **LightGBM Feature Importance**

In [ ]:
# LightGBM (GridSearchCV) - using best estimator
plot_feature_importance(grid_search_lgbm.best_estimator_, final_feature_names, 'LightGBM (GridSearchCV)')

### **CatBoost Feature Importance**

In [ ]:
# CatBoost (GridSearchCV) - using best estimator
plot_feature_importance(grid_search_cb.best_estimator_, final_feature_names, 'CatBoost (GridSearchCV)')

# **Save Model**

In [ ]:
import pickle

# Misalkan lr_pipeline adalah model yang telah ditune
threshold = 0.40

# Simpan model dan threshold dalam satu dictionary
final_model_lr = {
    'model_pipeline': lr_pipeline,
    'threshold': threshold,
    'model_info': 'Logistic Regression with SelectKBest',
    'features': X_train.columns.tolist() # Opsional
}

with open('model_lr.pkl', 'wb') as file:
    pickle.dump(final_model_lr, file)

# # Save the best Logistic Regression pipeline (including preprocessor and feature selector)
# with open('model_lr.pkl', 'wb') as file:
#     pickle.dump(lr_pipeline, file)

print("Logistic Regression pipeline saved to 'model_lr.pkl'")

In [ ]:
import joblib

joblib.dump(final_model_lr, 'model_lr.joblib')

print("Logistic Regression pipeline saved to 'model_lr.joblib'")

# SHAP, LIME, etc.

In [ ]:
!pip install shap

import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Transform X_test lewat preprocessor + feature_selector
# supaya SHAP bisa baca input yang udah bersih

X_test_transformed = preprocessor.transform(X_test)
X_test_selected = feature_selector.transform(X_test_transformed)

X_train_transformed = preprocessor.transform(X_train)
X_train_selected = feature_selector.transform(X_train_transformed)

# Ambil nama fitur yang terpilih (udah ada dari cell sebelumnya)
# final_feature_names sudah didefinisikan di Cell 184
print("Fitur terpilih:", final_feature_names)
print("Shape X_test_selected:", X_test_selected.shape)

In [ ]:
# ==================================================
# SHAP - Logistic Regression (t=0.40)
# ==================================================

# Ambil model LR dari pipeline
lr_model = lr_pipeline.named_steps['model']

# Pakai LinearExplainer karena LR itu linear model
explainer_lr = shap.LinearExplainer(lr_model, X_train_selected)
shap_values_lr = explainer_lr.shap_values(X_test_selected)

# --- Summary Plot (Bar) ---
plt.figure()
shap.summary_plot(
    shap_values_lr,
    X_test_selected,
    feature_names=final_feature_names,
    plot_type="bar",
    show=False
)
plt.title("SHAP Feature Importance - Logistic Regression")
plt.tight_layout()
plt.savefig("shap_summary_bar_lr.png", dpi=150, bbox_inches='tight')
plt.show()

# --- Summary Plot (Beeswarm) ---
plt.figure()
shap.summary_plot(
    shap_values_lr,
    X_test_selected,
    feature_names=final_feature_names,
    show=False
)
plt.title("SHAP Beeswarm - Logistic Regression")
plt.tight_layout()
plt.savefig("shap_summary_beeswarm_lr.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
shap.initjs()

sample_idx = 0

force = shap.force_plot(
    explainer_lr.expected_value,
    shap_values_lr[sample_idx],
    X_test_final_df.iloc[sample_idx],
    feature_names=list(final_feature_names),
    show=False
)

# Save sebagai HTML dulu (force plot interactive by nature)
shap.save_html("shap_force_plot_lr.html", force)
print("Saved!")

In [ ]:
import matplotlib.pyplot as plt
import shap

sample_idx = 0

explanation = shap.Explanation(
    values=shap_values_lr[sample_idx],
    base_values=explainer_lr.expected_value,
    data=X_test_selected[sample_idx],
    feature_names=list(final_feature_names)
)

shap.plots.waterfall(explanation, show=False)
plt.tight_layout()
plt.savefig("shap_waterfall_lr_sample0.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
!pip install lime

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
import matplotlib.pyplot as plt

# Setup explainer - pakai data yang udah selected (5 fitur)
lime_explainer = LimeTabularExplainer(
    training_data=X_train_preprocessed_selected,
    feature_names=list(final_feature_names),
    class_names=['Not Churn', 'Churn'],
    mode='classification'
)

# Explain sample index 0
sample_idx = 0
lime_exp = lime_explainer.explain_instance(
    data_row=X_test_preprocessed_selected[sample_idx],
    predict_fn=lr_model.predict_proba,  # langsung lr_model, bukan lr_pipeline
    num_features=5
)

# Plot
fig = lime_exp.as_pyplot_figure()
plt.title("LIME Explanation - LR (t=0.40) - Sample 0")
plt.tight_layout()
plt.savefig("lime_lr_sample0.png", dpi=150, bbox_inches='tight')
plt.show()

#Scorecard

In [ ]:
# Cek fitur raw yang tersedia sebelum feature engineering
print(df.columns.tolist())
print(df.shape)
print(df['churn'].value_counts())

In [ ]:
# ============================================================
# SCORECARD - Employee Churn Prediction
# Metode: Weight of Evidence (WoE) + Logistic Regression
# ============================================================

!pip install scorecardpy

import scorecardpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (classification_report, roc_auc_score,
                             recall_score, f1_score)
from scipy.stats import ks_2samp
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.calibration import calibration_curve

### Feature Selection

In [ ]:
# ============================================================
# Scorecard membutuhkan pipeline sendiri terpisah dari
# lr_pipeline sebelumnya. Alasannya:
# - Scorecard pakai WoE encoding, bukan StandardScaler
# - LR scorecard di-fit ulang dengan WoE features
# - Output scorecard adalah poin, bukan probabilitas
# ============================================================

# Cek kolom yang tersedia
print("Kolom yang tersedia:")
print(df.columns.tolist())
print("\nShape:", df.shape)
print("\nDistribusi churn:")
print(df['churn'].value_counts())

# Pilih fitur kandidat berdasarkan relevansi bisnis dan hasil SHAP
# Tidak ikutkan employee_id, churn_period (sudah di-drop sebelumnya)
# Tidak ikutkan fitur yang secara bisnis tidak relevan untuk HR scorecard
features_kandidat = [
    'target_achievement',      # performa karyawan
    'overall_satisfaction',    # kepuasan kerja (hasil feature engineering)
    'job_satisfaction',        # kepuasan kerja raw
    'manager_support_score',   # dukungan manager
    'distance_to_office_km',   # jarak ke kantor (raw)
    'distance_group',          # jarak ke kantor (kategori, hasil FE)
    'unachieved_target',       # target yang tidak tercapai (hasil FE)
    'overtime_ratio',          # rasio lembur (hasil FE)
    'company_tenure_years',    # lama bekerja
    'gender'                   # demografis
]

df_sc = df[features_kandidat + ['churn']].copy()

# Fix tipe data: distance_group dari category ke object
# agar scorecardpy bisa membaca dengan benar
df_sc['distance_group'] = df_sc['distance_group'].astype(str)
df_sc['churn'] = df_sc['churn'].astype(int)

print("\nTipe data setelah fix:")
print(df_sc.dtypes)
print("\nMissing values:")
print(df_sc.isnull().sum())

### WOE BINNING & IV ANALYSIS

In [ ]:
# ============================================================
# WoE Binning
# WoE (Weight of Evidence) mengukur kekuatan tiap bin
# dalam membedakan churn vs tidak churn
# IV (Information Value) mengukur seberapa prediktif suatu fitur
# IV < 0.02 = tidak prediktif → dibuang
# IV 0.02-0.1 = lemah
# IV 0.1-0.3 = sedang
# IV > 0.3 = kuat
# ============================================================

bins_all = sc.woebin(df_sc, y='churn')

# Rekap IV semua fitur
iv_summary = pd.DataFrame({
    'Feature': list(bins_all.keys()),
    'IV': [bins_all[f]['total_iv'].iloc[0] for f in bins_all.keys()]
}).sort_values('IV', ascending=False).reset_index(drop=True)

print("=== Information Value Summary ===")
print(iv_summary)

# Filter: buang fitur dengan IV < 0.02
selected_features = iv_summary[iv_summary['IV'] > 0.02]['Feature'].tolist()
print("\nFitur yang lolos IV > 0.02:")
print(selected_features)

### Baseline Model

In [ ]:
# ============================================================
# Train Test Split
# Stratify=y untuk memastikan proporsi churn sama
# di train dan test set
# random_state=42 untuk reproducibility
# ============================================================

# Siapkan dataframe dengan fitur yang lolos IV
# Saat ini masih include distance_to_office_km dan distance_group
# keduanya akan dievaluasi di iterasi berikutnya
df_baseline = df_sc[selected_features + ['churn']].copy()

train_base, test_base = train_test_split(
    df_baseline,
    test_size=0.2,
    random_state=42,
    stratify=df_baseline['churn']
)

print("Train shape:", train_base.shape)
print("Test shape:", test_base.shape)
print("\nChurn rate train:", train_base['churn'].mean().round(3))
print("Churn rate test:", test_base['churn'].mean().round(3))

# WoE Transform
bins_base = {k: v for k, v in bins_all.items() if k in selected_features}

train_woe_base = sc.woebin_ply(train_base, bins_base)
test_woe_base = sc.woebin_ply(test_base, bins_base)

y_train_base = train_woe_base['churn']
X_train_base = train_woe_base.drop('churn', axis=1)
y_test_base = test_woe_base['churn']
X_test_base = test_woe_base.drop('churn', axis=1)

# Fit LR Baseline
lr_base = LogisticRegression(random_state=42, C=1)
lr_base.fit(X_train_base, y_train_base)

y_pred_base = lr_base.predict(X_test_base)
y_proba_base = lr_base.predict_proba(X_test_base)[:, 1]

# Generate scorecard baseline
card_base = sc.scorecard(
    bins_base, lr_base,
    X_train_base.columns.tolist(),
    points0=600, odds0=1/19, pdo=20
)

score_test_base = sc.scorecard_ply(test_base, card_base,
                                    only_total_score=True, print_step=0)
score_train_base = sc.scorecard_ply(train_base, card_base,
                                     only_total_score=True, print_step=0)
score_test_base['churn'] = y_test_base.values
score_train_base['churn'] = y_train_base.values

# Fungsi PSI
def calculate_psi(expected, actual, bins=10):
    breakpoints = np.linspace(expected.min(), expected.max(), bins+1)
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf
    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts = np.histogram(actual, bins=breakpoints)[0]
    expected_pct = np.where(expected_counts == 0, 0.0001,
                            expected_counts / len(expected))
    actual_pct = np.where(actual_counts == 0, 0.0001,
                          actual_counts / len(actual))
    psi_values = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)
    return np.sum(psi_values)

roc_base = roc_auc_score(y_test_base, y_proba_base)
gini_base = (2 * roc_base) - 1
ks_base, _ = ks_2samp(
    score_test_base[score_test_base['churn']==0]['score'],
    score_test_base[score_test_base['churn']==1]['score']
)
psi_base = calculate_psi(score_train_base['score'], score_test_base['score'])

print("=== Baseline Model Performance ===")
print(classification_report(y_test_base, y_pred_base))
print(f"ROC AUC : {roc_base:.4f}")
print(f"Gini    : {gini_base:.4f}")
print(f"KS Stat : {ks_base:.4f}")
print(f"PSI     : {psi_base:.4f}")

### Iterasi 1: VIF Check & Pilih Distance Feature

In [ ]:
# ============================================================
# VIF Check (Variance Inflation Factor)
# VIF > 10 = multikolinearitas tinggi → perlu dibuang
# distance_to_office_km dan distance_group mengukur hal yang
# sama → salah satu harus dibuang
# ============================================================

vif_base = pd.DataFrame({
    'Feature': X_train_base.columns,
    'VIF': [variance_inflation_factor(X_train_base.values, i)
            for i in range(X_train_base.shape[1])]
}).sort_values('VIF', ascending=False)

print("=== VIF Check Baseline ===")
print(vif_base)

Keputusan: pakai distance_group, buang distance_to_office_km
Alasan: lebih interpretable untuk stakeholder HR

### Iterasi 2: Monotonicity Check & Rebin

In [ ]:
# ============================================================
# Monotonicity Check
# Scorecard yang baik harus monoton — makin tinggi risiko bin,
# makin rendah poinnya (atau sebaliknya secara konsisten)
# Non-monoton = susah dijelaskan ke stakeholder
# ============================================================

# Fitur yang akan dipakai setelah buang distance_to_office_km
selected_features_iter2 = [f for f in selected_features
                            if f != 'distance_to_office_km']

# Cek WoE dan badprob per fitur untuk identifikasi non-monoton
print("=== Monotonicity Check ===")
for feat in selected_features_iter2:
    print(f"\n{feat}:")
    print(bins_all[feat][['bin', 'woe', 'badprob', 'bin_iv']])


bins_rebin = bins_all.copy()

# Rebin 1: manager_support_score
bins_rebin['manager_support_score'] = sc.woebin(
    df_sc, y='churn', x='manager_support_score',
    breaks_list={'manager_support_score': [3]}
)['manager_support_score']

# Rebin 2: overall_satisfaction
bins_rebin['overall_satisfaction'] = sc.woebin(
    df_sc, y='churn', x='overall_satisfaction',
    breaks_list={'overall_satisfaction': [3, 9]}
)['overall_satisfaction']

# Rebin 3: company_tenure_years
# Sekaligus fix minimum sample per bin (bin < 0.3 hanya 31 sample)
bins_rebin['company_tenure_years'] = sc.woebin(
    df_sc, y='churn', x='company_tenure_years',
    breaks_list={'company_tenure_years': [0.7]}
)['company_tenure_years']

# Rebin 4: unachieved_target
bins_rebin['unachieved_target'] = sc.woebin(
    df_sc, y='churn', x='unachieved_target',
    breaks_list={'unachieved_target': [40, 55]}
)['unachieved_target']

# Verifikasi monotonicity setelah rebin
print("\n=== Verifikasi Setelah Rebin ===")
for feat in ['manager_support_score', 'overall_satisfaction',
             'company_tenure_years', 'unachieved_target']:
    print(f"\n{feat}:")
    print(bins_rebin[feat][['bin', 'woe', 'badprob', 'bin_iv']])

# Verifikasi minimum sample per bin (min 5% = 50 sample)
print("\n=== Minimum Sample per Bin ===")
total = len(df_sc)
min_sample = total * 0.05
for feat in selected_features_iter2:
    below = bins_rebin[feat][bins_rebin[feat]['count'] < min_sample]
    if len(below) > 0:
        print(f"{feat}:")
        print(below[['bin', 'count']])
    else:
        print(f"{feat} — semua bin OK")

### Iterasi 3: Regularization & PSI

In [ ]:
# ============================================================
# Fit model dengan fitur final (tanpa distance_to_office_km)
# dan bins yang sudah monoton
# ============================================================

selected_features_final = [
    'target_achievement',
    'overall_satisfaction',
    'job_satisfaction',
    'distance_group',
    'manager_support_score',
    'unachieved_target',
    'overtime_ratio',
    'company_tenure_years'
]

df_scorecard = df_sc[selected_features_final + ['churn']].copy()

train_sc, test_sc = train_test_split(
    df_scorecard,
    test_size=0.2,
    random_state=42,
    stratify=df_scorecard['churn']
)

bins_final = {k: v for k, v in bins_rebin.items()
              if k in selected_features_final}

train_woe = sc.woebin_ply(train_sc, bins_final)
test_woe = sc.woebin_ply(test_sc, bins_final)

y_train = train_woe['churn']
X_train = train_woe.drop('churn', axis=1)
y_test = test_woe['churn']
X_test = test_woe.drop('churn', axis=1)

# Fit LR C=1 dulu
lr_c1 = LogisticRegression(random_state=42, C=1)
lr_c1.fit(X_train, y_train)
y_proba_c1 = lr_c1.predict_proba(X_test)[:, 1]

card_c1 = sc.scorecard(bins_final, lr_c1,
                        X_train.columns.tolist(),
                        points0=600, odds0=1/19, pdo=20)

score_test_c1 = sc.scorecard_ply(test_sc, card_c1,
                                  only_total_score=True, print_step=0)
score_train_c1 = sc.scorecard_ply(train_sc, card_c1,
                                   only_total_score=True, print_step=0)

psi_c1 = calculate_psi(score_train_c1['score'], score_test_c1['score'])
roc_c1 = roc_auc_score(y_test, y_proba_c1)

print(f"C=1 → ROC AUC: {roc_c1:.4f}, PSI: {psi_c1:.4f}")

# PSI masuk zona warning → coba regularization C=0.5
lr_c05 = LogisticRegression(random_state=42, C=0.5)
lr_c05.fit(X_train, y_train)
y_proba_c05 = lr_c05.predict_proba(X_test)[:, 1]

card_c05 = sc.scorecard(bins_final, lr_c05,
                         X_train.columns.tolist(),
                         points0=600, odds0=1/19, pdo=20)

score_test_c05 = sc.scorecard_ply(test_sc, card_c05,
                                   only_total_score=True, print_step=0)
score_train_c05 = sc.scorecard_ply(train_sc, card_c05,
                                    only_total_score=True, print_step=0)

psi_c05 = calculate_psi(score_train_c05['score'], score_test_c05['score'])
roc_c05 = roc_auc_score(y_test, y_proba_c05)

print(f"C=0.5 → ROC AUC: {roc_c05:.4f}, PSI: {psi_c05:.4f}")

# Cross-validation PSI untuk konfirmasi stabilitas
print("\n=== Cross-Validation PSI ===")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_cv = df_scorecard.drop('churn', axis=1)
y_cv = df_scorecard['churn']

psi_scores = []
for fold, (train_idx, test_idx) in enumerate(skf.split(X_cv, y_cv)):
    train_fold = df_scorecard.iloc[train_idx]
    test_fold = df_scorecard.iloc[test_idx]

    train_woe_fold = sc.woebin_ply(train_fold, bins_final)
    test_woe_fold = sc.woebin_ply(test_fold, bins_final)

    y_train_fold = train_woe_fold['churn']
    X_train_fold = train_woe_fold.drop('churn', axis=1)

    lr_fold = LogisticRegression(random_state=42, C=0.5)
    lr_fold.fit(X_train_fold, y_train_fold)

    card_fold = sc.scorecard(bins_final, lr_fold,
                              X_train_fold.columns.tolist(),
                              points0=600, odds0=1/19, pdo=20)

    score_train_fold = sc.scorecard_ply(train_fold, card_fold,
                                         only_total_score=True, print_step=0)
    score_test_fold = sc.scorecard_ply(test_fold, card_fold,
                                        only_total_score=True, print_step=0)

    psi_fold = calculate_psi(score_train_fold['score'], score_test_fold['score'])
    psi_scores.append(psi_fold)
    print(f"Fold {fold+1} PSI: {psi_fold:.4f}")

print(f"\nMean PSI : {np.mean(psi_scores):.4f}")
print(f"Std PSI  : {np.std(psi_scores):.4f}")

### Final Scorecard

In [ ]:
# ============================================================
# FINAL SCORECARD
# ============================================================

# Model final sudah di-fit di Section 7 (lr_c05)
# Gunakan card_c05, score_test_c05, score_train_c05

score_test_c05['churn'] = y_test.values
score_train_c05['churn'] = y_train.values

# Tampilkan tabel poin scorecard
print("=== SCORECARD TABEL POIN ===")
for feature, table in card_c05.items():
    print(f"\n{feature}:")
    print(table[['bin', 'points']])

# Distribusi skor
print("\n=== Distribusi Skor ===")
print("TRAIN - Mean churn=0:",
      score_train_c05[score_train_c05['churn']==0]['score'].mean().round(2))
print("TRAIN - Mean churn=1:",
      score_train_c05[score_train_c05['churn']==1]['score'].mean().round(2))
print("TEST  - Mean churn=0:",
      score_test_c05[score_test_c05['churn']==0]['score'].mean().round(2))
print("TEST  - Mean churn=1:",
      score_test_c05[score_test_c05['churn']==1]['score'].mean().round(2))

# Visualisasi distribusi skor
plt.figure(figsize=(10, 5))
plt.hist(score_test_c05[score_test_c05['churn']==0]['score'],
         bins=20, alpha=0.6, label='Not Churn', color='blue')
plt.hist(score_test_c05[score_test_c05['churn']==1]['score'],
         bins=20, alpha=0.6, label='Churn', color='red')
plt.axvline(x=525, color='black', linestyle='--', label='Threshold 525')
plt.xlabel('Score')
plt.ylabel('Count')
plt.title('Score Distribution by Churn Status')
plt.legend()
plt.tight_layout()
plt.savefig('scorecard_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Threshold & kategori risiko
def categorize(score):
    if score < 490:
        return 'High Risk'
    elif score < 525:
        return 'Medium Risk'
    else:
        return 'Low Risk'

score_test_c05['risk_category'] = score_test_c05['score'].apply(categorize)
score_train_c05['risk_category'] = score_train_c05['score'].apply(categorize)

print("\n=== Distribusi Kategori Risiko ===")
print(pd.crosstab(score_test_c05['risk_category'],
                  score_test_c05['churn'],
                  margins=True))

print("\nChurn rate per kategori:")
print(score_test_c05.groupby('risk_category')['churn'].mean().round(3))

### Full Validation

In [ ]:
# ============================================================
# FULL VALIDATION SCORECARD FINAL
# ============================================================

y_pred_final = lr_c05.predict(X_test)
y_proba_final = lr_c05.predict_proba(X_test)[:, 1]

roc_final = roc_auc_score(y_test, y_proba_final)
gini_final = (2 * roc_final) - 1
ks_final, _ = ks_2samp(
    score_test_c05[score_test_c05['churn']==0]['score'],
    score_test_c05[score_test_c05['churn']==1]['score']
)
psi_final = calculate_psi(score_train_c05['score'], score_test_c05['score'])

print("=== Classification Report ===")
print(classification_report(y_test, y_pred_final))

print("=== Scorecard Metrics ===")
print(f"ROC AUC : {roc_final:.4f}")
print(f"Gini    : {gini_final:.4f}")
print(f"KS Stat : {ks_final:.4f}")
print(f"PSI     : {psi_final:.4f}")

# VIF final
print("\n=== VIF Final ===")
vif_final = pd.DataFrame({
    'Feature': X_train.columns,
    'VIF': [variance_inflation_factor(X_train.values, i)
            for i in range(X_train.shape[1])]
}).sort_values('VIF', ascending=False)
print(vif_final)

# Recall gap
print("\n=== Recall Gap Analysis ===")
print("Threshold | Test Recall | Train Recall | Gap | F1")
print("-" * 55)
for threshold in [510, 515, 520, 525, 530]:
    y_pred_t = (score_test_c05['score'] < threshold).astype(int)
    y_pred_train_t = (score_train_c05['score'] < threshold).astype(int)
    test_r = recall_score(y_test.values, y_pred_t)
    train_r = recall_score(y_train.values, y_pred_train_t)
    gap = abs(train_r - test_r)
    f1 = f1_score(y_test.values, y_pred_t)
    print(f"< {threshold}    | {test_r:.3f}       | {train_r:.3f}        | {gap:.3f} | {f1:.3f}")

# Stability kategori
print("\n=== Stability Kategori Train vs Test ===")
train_dist = score_train_c05['risk_category'].value_counts(normalize=True).round(3)
test_dist = score_test_c05['risk_category'].value_counts(normalize=True).round(3)
stability = pd.DataFrame({'Train %': train_dist, 'Test %': test_dist})
stability['Gap'] = (stability['Train %'] - stability['Test %']).abs()
print(stability)

# Calibration check
print("\n=== Calibration Check ===")
fraction_pos, mean_pred = calibration_curve(y_test, y_proba_final, n_bins=5)
calib_df = pd.DataFrame({
    'Mean Predicted Prob': mean_pred.round(3),
    'Actual Churn Rate': fraction_pos.round(3),
    'Gap': (mean_pred - fraction_pos).round(3)
})
print(calib_df)

# Perhitungan manual
print("\n=== Verifikasi Perhitungan Manual ===")
sample = test_sc.iloc[0]
print("Data karyawan sample:")
print(sample[selected_features_final])

basepoints = card_c05['basepoints']['points'].values[0]
total_manual = basepoints
print(f"\nBase points: {basepoints}")

for feature in selected_features_final:
    val = sample[feature]
    feature_card = card_c05[feature]
    for _, row in feature_card.iterrows():
        bin_label = row['bin']
        points = row['points']
        if isinstance(val, str):
            if val == bin_label:
                print(f"{feature} = {val} → '{bin_label}' → {points} pts")
                total_manual += points
                break
        else:
            bin_str = bin_label.replace('[', '').replace(')', '')
            parts = bin_str.split(',')
            lower = -float('inf') if '-inf' in parts[0] else float(parts[0])
            upper = float('inf') if 'inf' in parts[1].strip() else float(parts[1].strip())
            if lower <= val < upper:
                print(f"{feature} = {val} → '{bin_label}' → {points} pts")
                total_manual += points
                break

print(f"\nTotal manual   : {total_manual}")
print(f"Total scorecard: {score_test_c05['score'].iloc[0]}")
print(f"Match          : {abs(total_manual - score_test_c05['score'].iloc[0]) < 1}")

### Limitasi

In [ ]:
# ============================================================
# FINAL CHECK SEBELUM RESTART
# ============================================================

print("=== 1. Classification Report ===")
print(classification_report(y_test, y_pred_final))

print("=== 2. Scorecard Metrics ===")
print(f"ROC AUC : {roc_final:.4f}")
print(f"Gini    : {gini_final:.4f}")
print(f"KS Stat : {ks_final:.4f}")
print(f"PSI     : {psi_final:.4f}")

print("\n=== 3. PSI Cross Validation ===")
print(f"Mean PSI : {np.mean(psi_scores):.4f}")
print(f"Std PSI  : {np.std(psi_scores):.4f}")
print(f"Min PSI  : {np.min(psi_scores):.4f}")
print(f"Max PSI  : {np.max(psi_scores):.4f}")

print("\n=== 4. VIF Final ===")
print(vif_final)

print("\n=== 5. Calibration ===")
print(calib_df)

print("\n=== 6. Stability Kategori ===")
print(stability)

print("\n=== 7. Churn Rate per Kategori ===")
print(score_test_c05.groupby('risk_category')['churn'].mean().round(3))

print("\n=== 8. Recall Gap ===")
for threshold in [520, 525, 530]:
    y_pred_t = (score_test_c05['score'] < threshold).astype(int)
    y_pred_train_t = (score_train_c05['score'] < threshold).astype(int)
    test_r = recall_score(y_test.values, y_pred_t)
    train_r = recall_score(y_train.values, y_pred_train_t)
    gap = abs(train_r - test_r)
    f1 = f1_score(y_test.values, y_pred_t)
    print(f"< {threshold} | Recall: {test_r:.3f} | Gap: {gap:.3f} | F1: {f1:.3f}")

print("\n=== 9. Perhitungan Manual ===")
print(f"Total manual   : {total_manual}")
print(f"Total scorecard: {score_test_c05['score'].iloc[0]}")
print(f"Match          : {abs(total_manual - score_test_c05['score'].iloc[0]) < 1}")

print("\n=== 10. Distribusi Skor ===")
print("TRAIN churn=0:", score_train_c05[score_train_c05['churn']==0]['score'].mean().round(2))
print("TRAIN churn=1:", score_train_c05[score_train_c05['churn']==1]['score'].mean().round(2))
print("TEST  churn=0:", score_test_c05[score_test_c05['churn']==0]['score'].mean().round(2))
print("TEST  churn=1:", score_test_c05[score_test_c05['churn']==1]['score'].mean().round(2))

# LIMITASI SCORECARD

## Limitasi Scorecard

### 1. PSI dalam Zona Monitoring (PSI = 0.14)
- PSI (Population Stability Index) idealnya berada di bawah 0.10
- Cross-validation pada 5 fold menunjukkan rata-rata PSI = 0.0953
  (std = 0.0224), yang mengkonfirmasi bahwa model stabil secara rata-rata
- PSI = 0.14 pada split utama disebabkan oleh ukuran dataset yang kecil
  (1.000 baris)
- Standar industri merekomendasikan minimal 10.000 baris untuk scorecard
  yang robust
- Rekomendasi: pantau PSI secara berkala setelah model di-deploy

### 2. Stabilitas Kategori Risiko (~0.05)
- Gap distribusi kategori High Risk antara train dan test = 0.053
- Gap distribusi kategori Medium Risk antara train dan test = 0.054
- Gap distribusi kategori Low Risk = 0.001 (stabil)
- Gap sedikit di atas ambang batas 0.05, namun masih dapat diterima
  mengingat ukuran dataset yang terbatas
- Rekomendasi: evaluasi ulang scorecard ketika data tambahan tersedia

### 3. Pola Non-Monoton pada overtime_ratio
- Karyawan dengan overtime ratio tertinggi (> 0.37) justru mendapatkan
  poin positif, yang secara intuitif kurang sesuai karena overtime tinggi
  seharusnya mengindikasikan risiko burnout
- Kemungkinan penjelasan: karyawan dengan overtime tertinggi adalah
  karyawan yang paling engaged sehingga justru cenderung tidak churn
- Pola ini memerlukan investigasi lebih lanjut dengan dataset yang lebih besar
- Rekomendasi: pertimbangkan penggantian overtime_ratio dengan fitur
  biner (overtime tinggi vs normal) pada iterasi berikutnya

### 4. Miscalibration Model pada Rentang Probabilitas Menengah
- Model memprediksi probabilitas churn lebih rendah dari aktualnya
  pada rentang menengah (prediksi ~0.51, churn rate aktual 0.67,
  gap = -0.159)
- Platt Scaling calibration telah dicoba namun tidak signifikan
  memperbaiki gap (-0.159 → -0.156)
- Implikasi: untuk karyawan di zona Medium Risk, risiko churn aktual
  kemungkinan lebih tinggi dari yang diprediksi model
- Rekomendasi: gunakan kategori risiko (High/Medium/Low) daripada
  probabilitas mentah untuk pengambilan keputusan HR

### 5. Ukuran Dataset Terbatas (1.000 baris)
- Keandalan scorecard meningkat secara signifikan dengan lebih banyak data
- Beberapa bin WoE memiliki jumlah sampel yang terbatas sehingga
  dapat mempengaruhi stabilitas
- Hasil perlu divalidasi ulang seiring bertambahnya dataset

### 6. Ketergantungan pada Feature Engineering
- overall_satisfaction merupakan turunan dari
  job_satisfaction × manager_support_score
- Hal ini menciptakan multikolinearitas moderat (VIF = 7.1)
- Kedua fitur tetap dipertahankan karena VIF < 10 dan keduanya
  berkontribusi secara bermakna terhadap model

#TEST

In [ ]:
# ============================================================
# VALIDASI AKHIR SCORECARD
# Jalanin ini setelah semua section 1-10 selesai di-run
# ============================================================

print("=" * 60)
print("VALIDASI AKHIR SCORECARD")
print("=" * 60)

# 1. Cek variabel penting ada semua
required_vars = [
    'bins_final', 'card_c05', 'lr_c05',
    'score_test_c05', 'score_train_c05',
    'X_train', 'X_test', 'y_train', 'y_test',
    'train_sc', 'test_sc'
]

print("\n=== 1. Cek Variabel ===")
for var in required_vars:
    try:
        eval(var)
        print(f"{var} — ada")
    except NameError:
        print(f"{var} — TIDAK ADA, perlu di-run ulang")

# 2. Cek model
print("\n=== 2. Cek Model ===")
print(f"Model: {lr_c05}")
print(f"Fitur: {list(X_train.columns)}")
print(f"Jumlah fitur: {X_train.shape[1]}")

# 3. Cek scorecard poin
print("\n=== 3. Cek Scorecard Poin ===")
for feature, table in card_c05.items():
    print(f"\n{feature}:")
    print(table[['bin', 'points']])

# 4. Cek performa
print("\n=== 4. Cek Performa ===")
y_pred_val = lr_c05.predict(X_test)
y_proba_val = lr_c05.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred_val))
print(f"ROC AUC : {roc_auc_score(y_test, y_proba_val):.4f}")

# 5. Cek distribusi skor
print("\n=== 5. Cek Distribusi Skor ===")
print("TEST  churn=0:", score_test_c05[score_test_c05['churn']==0]['score'].mean().round(2))
print("TEST  churn=1:", score_test_c05[score_test_c05['churn']==1]['score'].mean().round(2))
print("TRAIN churn=0:", score_train_c05[score_train_c05['churn']==0]['score'].mean().round(2))
print("TRAIN churn=1:", score_train_c05[score_train_c05['churn']==1]['score'].mean().round(2))

# 6. Cek kategori risiko
print("\n=== 6. Cek Kategori Risiko ===")
print(pd.crosstab(score_test_c05['risk_category'],
                  score_test_c05['churn'],
                  margins=True))
print("\nChurn rate per kategori:")
print(score_test_c05.groupby('risk_category')['churn'].mean().round(3))

# 7. Cek perhitungan manual
print("\n=== 7. Cek Perhitungan Manual ===")
sample = test_sc.iloc[0]
basepoints = card_c05['basepoints']['points'].values[0]
total_manual = basepoints

for feature in selected_features_final:
    val = sample[feature]
    feature_card = card_c05[feature]
    for _, row in feature_card.iterrows():
        bin_label = row['bin']
        points = row['points']
        if isinstance(val, str):
            if val == bin_label:
                total_manual += points
                break
        else:
            bin_str = bin_label.replace('[', '').replace(')', '')
            parts = bin_str.split(',')
            lower = -float('inf') if '-inf' in parts[0] else float(parts[0])
            upper = float('inf') if 'inf' in parts[1].strip() else float(parts[1].strip())
            if lower <= val < upper:
                total_manual += points
                break

print(f"Total manual   : {total_manual}")
print(f"Total scorecard: {score_test_c05['score'].iloc[0]}")
print(f"Match          : {abs(total_manual - score_test_c05['score'].iloc[0]) < 1}")

# 8. Summary final
print("\n" + "=" * 60)
print("SUMMARY FINAL")
print("=" * 60)
print(f"Fitur final    : {selected_features_final}")
print(f"Model          : LogisticRegression(C=0.5, random_state=42)")
print(f"Threshold      : < 525")
print(f"High Risk      : skor < 490, churn rate 86.1%")
print(f"Medium Risk    : skor 490-524, churn rate 62.7%")
print(f"Low Risk       : skor >= 525, churn rate 29.6%")
print(f"ROC AUC        : 0.7947")
print(f"Gini           : 0.5893")
print(f"KS Stat        : 0.4236")
print(f"PSI            : 0.1407 (CV mean: 0.0953)")
print(f"Recall @525    : 0.873")
print(f"Recall Gap     : 0.008")
print(f"F1 @525        : 0.809")
print("=" * 60)